# 11. Final Test + Slice/Error Analysis
## 2024-25 → 2025-26 Locked Holdout

10번 안정성 점검에서 **FULL feature set 유지**가 확정된 뒤,
이번 Notebook에서 처음으로 Final Test를 엽니다.

---

# 이미 동결된 구조

```text
Label
→ 09-01A conservative audited labels (development only)

Model A
→ Fixed CatBoost Classifier
→ S3 + transfer_event_preseason + destination_in_big5
→ threshold는 development 마지막 시즌에서 선택

Model B
→ Fixed CatBoost Regressor S3
→ matched_next=True development rows만 학습

Integration
→ Hard Gate

P(Big5 presence) < threshold
→ 0 goals

P(Big5 presence) >= threshold
→ Model B conditional goal prediction
```

10번 이후에는 feature / model / threshold 규칙을 더 이상 선택하지 않습니다.

---

# Final Test 규칙

```text
Input season : 2024-2025
Target season: 2025-2026
```

중요:

1. Test label은 feature 생성 동안 숨깁니다.
2. Model A / B 학습과 threshold 결정이 끝난 뒤 prediction을 먼저 고정합니다.
3. 그 다음에만 Test label을 다시 붙여 최종 평가합니다.
4. Test 결과를 보고 model을 다시 선택하지 않습니다.

---

# Test label 주의

Development에서는 09-01A에서 명확한 label contradiction 20건을 보정했습니다.

Final Test는 locked holdout이므로
**Test 결과를 본 뒤 추가 label correction을 하지 않습니다.**

따라서 Test CSV에 원래 저장된 `matched_next / next_goals`를
frozen ground truth로 평가하고,
label matching noise 가능성은 최종 한계로 별도 기록합니다.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 0. Google Drive Setup

브라우저 Google Colab이면 먼저:

```python
from google.colab import drive
drive.mount('/content/drive')
```

이미 마운트되어 있으면 다시 할 필요 없습니다.

In [4]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive가 마운트되어 있지 않습니다. "
        "drive.mount('/content/drive')를 먼저 실행하세요."
    )

print(
    "Google Drive mounted:",
    DRIVE_ROOT.exists(),
)

Google Drive mounted: True


In [5]:
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import random
import re
import unicodedata
import warnings
import zipfile

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------------
# 정책 설정
# ------------------------------------------------------------------

ALLOW_FUZZY_PLAYER_AUTO_MATCH = False

FALLBACK_CUTOFF_MONTH = 7
FALLBACK_CUTOFF_DAY = 31

MIN_PLAYER_ROW_MATCH_RATE = 0.80
MIN_OLD_TEAM_MATCH_RATE = 0.98

# Transfer event ↔ DB transfer date pair matching
TRANSFER_PAIR_MIN_SCORE = 0.60
TRANSFER_PAIR_STRONG_SCORE = 0.78

# 현재 팀 상태 ↔ transfer event의 from/to club 연결 기준
TEAM_CONTEXT_MIN_SCORE = 0.66

# final destination이 현재 팀과 같은 팀인지 판단
SAME_CLUB_SCORE = 0.82

# 새 팀 ↔ team_season_stats match
NEW_TEAM_MIN_SCORE = 0.64
NEW_TEAM_MIN_MARGIN = 0.04

MANUAL_PLAYER_ID_OVERRIDES = {
    # "Example Player": 123456,
}

MANUAL_DESTINATION_TEAM_OVERRIDES = {
    # "Paris Saint-Germain FC": "Paris S-G",
}

BIG5_LEAGUES = [
    "Premier League",
    "La Liga",
    "Bundesliga",
    "Serie A",
    "Ligue 1",
]

BIG5_COMPETITION_IDS = {
    "Premier League": "GB1",
    "La Liga": "ES1",
    "Bundesliga": "L1",
    "Serie A": "IT1",
    "Ligue 1": "FR1",
}

COMPETITION_ID_TO_BIG5_LEAGUE = {
    v: k for k, v in BIG5_COMPETITION_IDS.items()
}

LEAGUE_COUNTRY = {
    "Premier League": "England",
    "La Liga": "Spain",
    "Bundesliga": "Germany",
    "Serie A": "Italy",
    "Ligue 1": "France",
}

ZIP_FOLDER_TO_LEAGUE = {
    "premier_league": "Premier League",
    "laliga": "La Liga",
    "bundesliga": "Bundesliga",
    "serie_a": "Serie A",
    "ligue_1": "Ligue 1",
}

# eordo / Transfermarkt의 리그 표기 차이를 한 번에 통일
LEAGUE_ALIASES = {
    "premier league": "Premier League",
    "premierleague": "Premier League",
    "england premier league": "Premier League",

    "la liga": "La Liga",
    "laliga": "La Liga",
    "la liga ea sports": "La Liga",

    "bundesliga": "Bundesliga",
    "1 bundesliga": "Bundesliga",

    "serie a": "Serie A",
    "seriea": "Serie A",

    "ligue 1": "Ligue 1",
    "ligue1": "Ligue 1",
}

# Club 이름 비교 시 의미가 약한 일반 토큰
GENERIC_CLUB_TOKENS = {
    "fc", "cf", "ac", "afc", "sc", "ss", "bc", "sv", "vfl", "vfb",
    "club", "football", "futbol", "calcio", "de", "the"
}

# 자주 등장하는 축약 표기
CLUB_NAME_ALIASES = {
    "manchester utd": "manchester united",
    "man utd": "manchester united",
    "manchester city": "manchester city",
    "man city": "manchester city",
    "paris s g": "paris saint germain",
    "paris sg": "paris saint germain",
    "psg": "paris saint germain",
    "eint frankfurt": "eintracht frankfurt",
    "ein frankfurt": "eintracht frankfurt",
    "nott ham forest": "nottingham forest",
    "nottm forest": "nottingham forest",
    "dortmund": "borussia dortmund",
    "gladbach": "borussia monchengladbach",
    "m gladbach": "borussia monchengladbach",
    "leverkusen": "bayer leverkusen",
    "bayern munich": "bayern munich",
}

print("11 external-feature settings loaded.")

11 external-feature settings loaded.


## 1. DuckDB / packages

In [6]:
import subprocess
import sys

try:
    import duckdb

except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "duckdb",
        ]
    )

    import duckdb

print(
    "duckdb:",
    duckdb.__version__,
)

duckdb: 1.3.2


# Part A. Final Test input 탐색

이전 `(1)` 폴더 문제를 피하기 위해:

```text
next_season_goal_prediction
next_season_goal_prediction (1)
next_season_goal_prediction (2)
...
```

중 `09_01A_snapshot_conservative_labels.csv`가 있는 프로젝트를 우선 선택합니다.

In [7]:
AUDITED_DEV_FILENAME = (
    "09_01A_snapshot_conservative_labels.csv"
)


def candidate_project_roots():
    roots = []

    for p in (
        DRIVE_ROOT.glob(
            "next_season_goal_prediction*"
        )
    ):
        if p.is_dir():
            roots.append(
                p
            )

    # 현재 runtime/project가 별도로 잡힌 경우도 보조
    for p in [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]:
        if (
            p.exists()
            and p.is_dir()
        ):
            roots.append(
                p
            )

    unique = []

    seen = set()

    for p in roots:
        try:
            rp = p.resolve()
        except Exception:
            rp = p

        key = str(
            rp
        )

        if key not in seen:
            seen.add(
                key
            )
            unique.append(
                rp
            )

    return unique


PROJECT_ROOTS = (
    candidate_project_roots()
)


def find_artifact_project():
    candidates = []

    for root in (
        PROJECT_ROOTS
    ):
        artifact_dir = (
            root
            / "artifacts"
        )

        audited = (
            artifact_dir
            / AUDITED_DEV_FILENAME
        )

        if audited.exists():
            candidates.append(
                (
                    audited.stat().st_mtime,
                    root,
                    artifact_dir,
                )
            )

    if not candidates:
        # artifacts가 한 단계 더 아래에 있을 수 있어 제한적으로 검색
        for root in (
            PROJECT_ROOTS
        ):
            try:
                for audited in (
                    root.rglob(
                        AUDITED_DEV_FILENAME
                    )
                ):
                    if (
                        audited.parent.name
                        != "artifacts"
                    ):
                        continue

                    candidates.append(
                        (
                            audited.stat().st_mtime,
                            audited.parent.parent,
                            audited.parent,
                        )
                    )
            except (
                PermissionError,
                OSError,
            ):
                continue

    if not candidates:
        raise FileNotFoundError(
            f"{AUDITED_DEV_FILENAME}가 있는 "
            "프로젝트를 찾지 못했습니다."
        )

    candidates.sort(
        key=lambda x:
            x[0],
        reverse=True,
    )

    return (
        candidates[0][1],
        candidates[0][2],
    )


PROJECT_ROOT, ARTIFACT_DIR = (
    find_artifact_project()
)

AUDITED_DEV_PATH = (
    ARTIFACT_DIR
    / AUDITED_DEV_FILENAME
)


def first_existing(
    candidates,
):
    for path in candidates:
        if (
            path is not None
            and Path(
                path
            ).exists()
        ):
            return Path(
                path
            )

    return None


def recursive_find_file(
    roots,
    filename,
):
    hits = []

    for root in roots:
        try:
            for p in (
                root.rglob(
                    filename
                )
            ):
                if p.is_file():
                    hits.append(
                        p
                    )
        except (
            PermissionError,
            OSError,
        ):
            continue

    if not hits:
        return None

    hits.sort(
        key=lambda p: (
            len(
                str(
                    p
                )
            ),
            str(
                p
            ),
        )
    )

    return hits[0]


DATA_ROOT_CANDIDATES = [
    PROJECT_ROOT
    / "data",
    PROJECT_ROOT.parent
    / "data",
]

TRAIN_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "long_basic"
    / "train.csv",
])

VALIDATION_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "long_basic"
    / "validation.csv",
])

TEST_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "long_basic"
    / "test.csv",
])

TRANSFER_DATA_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "transfermarkt-data-master",
])

TRANSFER_DUCKDB_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "transfermarkt-datasets.duckdb",
])

TEAM_STATS_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "team_season_stats_2000_2024.csv",
])

TEAM_ALIAS_PATH = first_existing([
    PROJECT_ROOT
    / "data"
    / "team_name_alias_map.csv",
])


# 직접 경로가 없으면 프로젝트 roots 안에서 fallback search
if TRAIN_PATH is None:
    TRAIN_PATH = recursive_find_file(
        PROJECT_ROOTS,
        "train.csv",
    )

if VALIDATION_PATH is None:
    VALIDATION_PATH = recursive_find_file(
        PROJECT_ROOTS,
        "validation.csv",
    )

if TEST_PATH is None:
    # long_basic/test.csv를 우선 찾기 위한 별도 후보
    test_hits = []

    for root in (
        PROJECT_ROOTS
    ):
        try:
            for p in root.rglob(
                "test.csv"
            ):
                if (
                    "long_basic"
                    in p.parts
                ):
                    test_hits.append(
                        p
                    )
        except (
            PermissionError,
            OSError,
        ):
            continue

    if test_hits:
        test_hits.sort(
            key=lambda p:
                len(
                    str(
                        p
                    )
                )
        )

        TEST_PATH = (
            test_hits[0]
        )


def find_dir(
    roots,
    dirname,
):
    for root in roots:
        direct = (
            root
            / "data"
            / dirname
        )

        if direct.exists():
            return direct

    for root in roots:
        try:
            for p in root.rglob(
                dirname
            ):
                if p.is_dir():
                    return p
        except (
            PermissionError,
            OSError,
        ):
            continue

    return None


if TRANSFER_DATA_PATH is None:
    TRANSFER_DATA_PATH = find_dir(
        PROJECT_ROOTS,
        "transfermarkt-data-master",
    )

if TRANSFER_DUCKDB_PATH is None:
    TRANSFER_DUCKDB_PATH = recursive_find_file(
        PROJECT_ROOTS,
        "transfermarkt-datasets.duckdb",
    )

if TEAM_STATS_PATH is None:
    TEAM_STATS_PATH = recursive_find_file(
        PROJECT_ROOTS,
        "team_season_stats_2000_2024.csv",
    )

if TEAM_ALIAS_PATH is None:
    TEAM_ALIAS_PATH = recursive_find_file(
        PROJECT_ROOTS,
        "team_name_alias_map.csv",
    )


required_paths = {
    "AUDITED DEV": (
        AUDITED_DEV_PATH
    ),
    "TRAIN": (
        TRAIN_PATH
    ),
    "VALIDATION": (
        VALIDATION_PATH
    ),
    "TEST": (
        TEST_PATH
    ),
    "TM DATA": (
        TRANSFER_DATA_PATH
    ),
    "TM DUCKDB": (
        TRANSFER_DUCKDB_PATH
    ),
    "TEAM STATS": (
        TEAM_STATS_PATH
    ),
    "TEAM ALIAS": (
        TEAM_ALIAS_PATH
    ),
}

for name, path in (
    required_paths.items()
):
    print(
        f"{name:<12}:",
        path,
    )

missing = [
    name
    for name, path
    in required_paths.items()
    if (
        path is None
        or not Path(
            path
        ).exists()
    )
]

if missing:
    raise FileNotFoundError(
        "필수 입력을 찾지 못했습니다: "
        + ", ".join(
            missing
        )
    )

AUDITED DEV : /content/drive/MyDrive/next_season_goal_prediction/artifacts/09_01A_snapshot_conservative_labels.csv
TRAIN       : /content/drive/MyDrive/next_season_goal_prediction/data/long_basic/train.csv
VALIDATION  : /content/drive/MyDrive/next_season_goal_prediction/data/long_basic/validation.csv
TEST        : /content/drive/MyDrive/next_season_goal_prediction/data/long_basic/test.csv
TM DATA     : /content/drive/MyDrive/next_season_goal_prediction/data/transfermarkt-data-master
TM DUCKDB   : /content/drive/MyDrive/next_season_goal_prediction/data/transfermarkt-datasets.duckdb
TEAM STATS  : /content/drive/MyDrive/next_season_goal_prediction/data/team_season_stats_2000_2024.csv
TEAM ALIAS  : /content/drive/MyDrive/next_season_goal_prediction/data/team_name_alias_map.csv


## 2. Train / Validation / Test 로드 — Test label 즉시 숨김

Test label은 별도 `final_test_labels`에 보관하고,
feature integration용 Test frame에서는 `NaN`으로 바꿉니다.

따라서 외부 feature 생성 / crosswalk / cutoff 계산 과정이
Final Test 정답을 사용할 수 없습니다.

In [8]:
train_raw = pd.read_csv(
    TRAIN_PATH
)

validation_raw = pd.read_csv(
    VALIDATION_PATH
)

test_raw = pd.read_csv(
    TEST_PATH
)

team_stats = pd.read_csv(
    TEAM_STATS_PATH
)

team_alias = pd.read_csv(
    TEAM_ALIAS_PATH
)


for frame in [
    train_raw,
    validation_raw,
    test_raw,
    team_stats,
    team_alias,
]:
    drop_cols = [
        c
        for c in frame.columns
        if (
            c.lower().startswith(
                "unnamed"
            )
            or c == "index"
        )
    ]

    if drop_cols:
        frame.drop(
            columns=drop_cols,
            inplace=True,
        )


assert (
    set(
        test_raw[
            "season"
        ].astype(str)
    )
    == {
        "2024-2025",
    }
), (
    "Final Test input season이 "
    "2024-2025가 아닙니다."
)


# concat 전 row count를 저장
DEV_RAW_N = (
    len(
        train_raw
    )
    + len(
        validation_raw
    )
)


all_raw = pd.concat(
    [
        train_raw,
        validation_raw,
        test_raw,
    ],
    ignore_index=True,
).copy()


all_raw[
    "season_start"
] = (
    all_raw[
        "season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

all_raw[
    "target_year"
] = (
    all_raw[
        "target_season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

all_raw[
    "row_id"
] = np.arange(
    len(
        all_raw
    ),
    dtype=int,
)


test_mask_raw = (
    all_raw[
        "season"
    ]
    .astype(str)
    .eq(
        "2024-2025"
    )
)

final_test_labels = (
    all_raw.loc[
        test_mask_raw,
        [
            "row_id",
            "player",
            "team",
            "league",
            "season",
            "target_season",
            "matched_next",
            "next_goals",
            "next_10plus",
        ],
    ]
    .copy()
    .rename(
        columns={
            "matched_next": (
                "final_matched_next"
            ),
            "next_goals": (
                "final_next_goals"
            ),
            "next_10plus": (
                "final_next_10plus"
            ),
        }
    )
)


# Final Test label 숨기기
for col in [
    "matched_next",
    "next_goals",
    "next_10plus",
]:
    if col in all_raw.columns:
        all_raw.loc[
            test_mask_raw,
            col,
        ] = np.nan


# 08 integration 코드와 변수명 호환
dev = all_raw.copy()


print(
    "Train:",
    train_raw.shape,
)

print(
    "Validation:",
    validation_raw.shape,
)

print(
    "Final Test:",
    test_raw.shape,
)

print(
    "Combined feature-build rows:",
    dev.shape,
)

print(
    "✅ Final Test labels hidden:",
    dev.loc[
        test_mask_raw,
        "next_goals",
    ].isna().all(),
)

Train: (22430, 22)
Validation: (923, 40)
Final Test: (926, 40)
Combined feature-build rows: (24279, 43)
✅ Final Test labels hidden: True


## 4. 문자열 정규화 함수

외부 데이터 연결에서 가장 흔한 문제는 표기 차이입니다.

예:

```text
Kylian Mbappé ↔ Kylian Mbappe
Paris Saint-Germain ↔ Paris S-G
1. FC Köln ↔ Köln
```

선수 이름은 악센트·문장부호를 제거한 정규화 키를 만듭니다.

클럽은 지나치게 공격적으로 단어를 삭제하면 서로 다른 클럽이 합쳐질 수 있으므로
기본 정규화 + fuzzy similarity를 함께 사용합니다.

In [9]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()

    text = unicodedata.normalize(
        "NFKD",
        text,
    )

    text = "".join(
        ch
        for ch in text
        if not unicodedata.combining(ch)
    )

    replacements = {
        "ß": "ss",
        "ø": "o",
        "đ": "d",
        "ł": "l",
        "ð": "d",
        "þ": "th",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def canonicalize_league(value):
    if pd.isna(value):
        return np.nan

    key = normalize_text(value)

    return LEAGUE_ALIASES.get(
        key,
        str(value).strip(),
    )


def normalize_club(value):
    text = normalize_text(value)

    if not text:
        return ""

    text = CLUB_NAME_ALIASES.get(
        text,
        text,
    )

    return text


def club_core_tokens(value):
    text = normalize_club(value)

    tokens = [
        token
        for token in text.split()
        if token not in GENERIC_CLUB_TOKENS
        and not token.isdigit()
    ]

    return tokens


def string_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def club_similarity(a, b):
    """
    일반 문자열 similarity보다 club 표기에 강한 비교 함수.

    예:
    - Getafe CF ↔ Getafe
    - Olympique Marseille ↔ Marseille
    - Borussia Dortmund ↔ Dortmund
    """
    a_norm = normalize_club(a)
    b_norm = normalize_club(b)

    if not a_norm or not b_norm:
        return 0.0

    if a_norm == b_norm:
        return 1.0

    base = SequenceMatcher(
        None,
        a_norm,
        b_norm,
    ).ratio()

    a_tokens = club_core_tokens(a)
    b_tokens = club_core_tokens(b)

    if not a_tokens or not b_tokens:
        return base

    a_set = set(a_tokens)
    b_set = set(b_tokens)

    # FC / CF / AC 같은 일반 토큰 제거 후 동일
    if a_set == b_set:
        return max(base, 0.98)

    # Marseille ↔ Olympique Marseille,
    # Dortmund ↔ Borussia Dortmund 같은 관계
    smaller = (
        a_set if len(a_set) <= len(b_set)
        else b_set
    )
    larger = (
        b_set if len(a_set) <= len(b_set)
        else a_set
    )

    distinctive = [
        t
        for t in smaller
        if len(t) >= 5
    ]

    if (
        distinctive
        and smaller.issubset(larger)
    ):
        base = max(base, 0.92)

    return float(base)


dev["norm_player"] = (
    dev["player"]
    .map(normalize_text)
)

dev["norm_current_team"] = (
    dev["team"]
    .map(normalize_club)
)

print(
    dev[
        [
            "player",
            "norm_player",
            "team",
            "norm_current_team",
        ]
    ].head()
)

                player          norm_player            team norm_current_team
0  Abdelhafid Tasfaout  abdelhafid tasfaout        Guingamp          guingamp
1        Abder Ramdane        abder ramdane        Freiburg          freiburg
2             Adaílton             adailton   Hellas Verona     hellas verona
3         Ade Akinbiyi         ade akinbiyi  Leicester City    leicester city
4         Adel Sellimi         adel sellimi        Freiburg          freiburg


# Part A. Transfermarkt 원본 데이터 확인

## 5. eordo Transfermarkt ZIP 로드

받은 ZIP에서 Big 5의 2000~2025 시즌 CSV를 직접 읽습니다.

이 데이터의 주요 장점:

- `player_id`
- summer / winter
- in / out
- 이적 당시 시장가치
- dealing club / country
- fee
- loan

ZIP의 `season=2024`는 **2024-25 시즌**을 의미합니다.

In [10]:
def load_eordo_big5_folder(
    root_path,
    start_year=2000,
    end_year=2025,
):
    frames = []

    root_path = Path(root_path)

    for folder, expected_league in ZIP_FOLDER_TO_LEAGUE.items():
        league_dir = root_path / folder

        if not league_dir.exists():
            print(
                f"[WARNING] 폴더 없음: {league_dir}"
            )
            continue

        for year in range(
            start_year,
            end_year + 1,
        ):
            csv_path = (
                league_dir
                / f"{year}.csv"
            )

            if not csv_path.exists():
                continue

            tmp = pd.read_csv(
                csv_path
            )

            tmp[
                "source_folder"
            ] = folder

            tmp[
                "source_file"
            ] = str(csv_path)

            frames.append(tmp)

    if not frames:
        raise ValueError(
            "Big5 Transfermarkt CSV를 찾지 못했습니다."
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


eordo = load_eordo_big5_folder(
    TRANSFER_DATA_PATH,
    start_year=2000,
    end_year=2025,
)

eordo["player_id"] = pd.to_numeric(
    eordo["player_id"],
    errors="coerce",
).astype("Int64")

eordo["season"] = pd.to_numeric(
    eordo["season"],
    errors="coerce",
).astype("Int64")

eordo["norm_player"] = (
    eordo["player_name"]
    .map(normalize_text)
)

# v2 핵심: Laliga → La Liga 등 표기 통일
eordo[
    "league_raw"
] = eordo["league"]

eordo[
    "league"
] = (
    eordo["league"]
    .map(canonicalize_league)
)

league_normalization_changes = (
    eordo["league_raw"]
    .astype(str)
    .ne(
        eordo["league"]
        .astype(str)
    )
    .sum()
)

print(
    "Rows:",
    len(eordo),
)

print(
    "Unique player IDs:",
    eordo["player_id"].nunique(),
)

print(
    "Season:",
    eordo["season"].min(),
    "~",
    eordo["season"].max(),
)

print(
    "League labels normalized:",
    int(league_normalization_changes),
)

print(
    "Canonical leagues:",
    sorted(
        eordo["league"]
        .dropna()
        .astype(str)
        .unique()
    )[:20],
)

display(
    eordo[
        [
            "season",
            "league_raw",
            "league",
            "club",
            "window",
            "movement",
            "player_name",
            "player_id",
            "market_value",
            "dealing_club",
            "fee",
            "is_loan",
        ]
    ].head()
)

Rows: 78460
Unique player IDs: 21154
Season: 2000 ~ 2025
League labels normalized: 12992
Canonical leagues: ['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']


,season,league_raw,league,club,window,movement,player_name,player_id,market_value,dealing_club,fee,is_loan
0,2000,Premier League,Premier League,Arsenal FC,summer,in,Sylvain Wiltord,3188,NaN,FC Girondins Bordeaux,17500000.0,0
1,2000,Premier League,Premier League,Arsenal FC,summer,in,Francis Jeffers,3186,NaN,Everton FC,15300000.0,0
2,2000,Premier League,Premier League,Arsenal FC,summer,in,Laurén,3189,NaN,RCD Mallorca,10700000.0,0
3,2000,Premier League,Premier League,Arsenal FC,summer,in,Robert Pirès,3185,NaN,Olympique Marseille,9800000.0,0
4,2000,Premier League,Premier League,Arsenal FC,summer,in,Igors Stepanovs,3200,NaN,Skonto Riga (- 2016),1500000.0,0


## 6. eordo 데이터 기본 품질 확인

In [11]:
eordo_quality = pd.DataFrame({
    "metric": [
        "all_rows",
        "unique_player_ids",
        "summer_rows",
        "summer_in_rows",
        "summer_out_rows",
        "market_value_nonnull_rate_summer",
        "fee_nonnull_rate_summer",
    ],
    "value": [
        len(eordo),
        eordo["player_id"].nunique(),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            .sum()
        ),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            & eordo["movement"]
            .astype(str)
            .str.lower()
            .eq("in")
        ).sum(),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            & eordo["movement"]
            .astype(str)
            .str.lower()
            .eq("out")
        ).sum(),
        eordo.loc[
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer"),
            "market_value",
        ].notna().mean(),
        eordo.loc[
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer"),
            "fee",
        ].notna().mean(),
    ],
})

eordo_quality

,metric,value
0,all_rows,78460.000000
1,unique_player_ids,21154.000000
2,summer_rows,60759.000000
3,summer_in_rows,20924.000000
4,summer_out_rows,39835.000000
5,market_value_nonnull_rate_summer,0.800556
6,fee_nonnull_rate_summer,0.881285


## 7. DuckDB 테이블과 스키마 확인

`transfermarkt-datasets.duckdb`는:

- players
- player_valuations
- transfers
- games

를 주로 사용합니다.

여기서는 실제 파일의 column을 출력해서
나중에 데이터 버전이 바뀌더라도 바로 확인할 수 있게 합니다.

In [12]:
con = duckdb.connect(
    str(TRANSFER_DUCKDB_PATH),
    read_only=True,
)

tables = (
    con.execute("SHOW TABLES")
    .df()
)

display(tables)

REQUIRED_DUCKDB_TABLES = {
    "players",
    "player_valuations",
    "transfers",
    "games",
}

available_tables = set(
    tables.iloc[:, 0].astype(str)
)

missing_tables = (
    REQUIRED_DUCKDB_TABLES
    - available_tables
)

assert not missing_tables, (
    f"DuckDB 필수 테이블 누락: {missing_tables}"
)

for table in [
    "players",
    "player_valuations",
    "transfers",
    "games",
]:
    print("\n", "=" * 70)
    print(table)
    print("=" * 70)

    display(
        con.execute(
            f"DESCRIBE {table}"
        ).df()
    )

,name
0,appearances
1,club_games
2,clubs
3,competitions
4,countries
5,game_events
6,game_lineups
7,games
8,national_teams
9,player_valuations



players


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,last_season,VARCHAR,YES,None,None,None
5,current_club_id,VARCHAR,YES,None,None,None
6,player_code,VARCHAR,YES,None,None,None
7,country_of_birth,VARCHAR,YES,None,None,None
8,city_of_birth,VARCHAR,YES,None,None,None
9,country_of_citizenship,VARCHAR,YES,None,None,None



player_valuations


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,date,DATE,YES,None,None,None
2,market_value_in_eur,INTEGER,YES,None,None,None
3,current_club_name,VARCHAR,YES,None,None,None
4,current_club_id,INTEGER,YES,None,None,None
5,player_club_domestic_competition_id,VARCHAR,YES,None,None,None



transfers


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,transfer_date,DATE,YES,None,None,None
2,transfer_season,VARCHAR,YES,None,None,None
3,from_club_id,INTEGER,YES,None,None,None
4,to_club_id,INTEGER,YES,None,None,None
5,from_club_name,VARCHAR,YES,None,None,None
6,to_club_name,VARCHAR,YES,None,None,None
7,transfer_fee,"DECIMAL(18,3)",YES,None,None,None
8,market_value_in_eur,"DECIMAL(18,3)",YES,None,None,None
9,player_name,VARCHAR,YES,None,None,None



games


,column_name,column_type,null,key,default,extra
0,game_id,VARCHAR,YES,None,None,None
1,competition_id,VARCHAR,YES,None,None,None
2,season,VARCHAR,YES,None,None,None
3,round,VARCHAR,YES,None,None,None
4,date,DATE,YES,None,None,None
5,home_club_id,INTEGER,YES,None,None,None
6,away_club_id,INTEGER,YES,None,None,None
7,home_club_goals,INTEGER,YES,None,None,None
8,away_club_goals,INTEGER,YES,None,None,None
9,home_club_position,INTEGER,YES,None,None,None


## 8. DuckDB 필요 테이블 로드

현재 파일 규모에서는 필요한 4개 테이블만 pandas로 읽어도 충분합니다.

`appearances`, `game_events` 같은 대용량 테이블은 이번 단계에 필요하지 않습니다.

In [13]:
tm_players_raw = (
    con.execute(
        "SELECT * FROM players"
    ).df()
)

valuations_raw = (
    con.execute(
        "SELECT * FROM player_valuations"
    ).df()
)

db_transfers_raw = (
    con.execute(
        "SELECT * FROM transfers"
    ).df()
)

games_raw = (
    con.execute(
        "SELECT * FROM games"
    ).df()
)

print(
    "players          :",
    tm_players_raw.shape,
)
print(
    "player_valuations:",
    valuations_raw.shape,
)
print(
    "transfers        :",
    db_transfers_raw.shape,
)
print(
    "games            :",
    games_raw.shape,
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

players          : (50149, 26)
player_valuations: (656301, 6)
transfers        : (175165, 10)
games            : (88958, 23)


## 9. Column resolver

Transfermarkt dataset 버전에 따라 날짜 column 등이 조금 달라질 가능성에 대비해서
후보 이름 중 실제 존재하는 column을 자동으로 선택합니다.

In [14]:
def resolve_col(
    df,
    candidates,
    required=True,
):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise KeyError(
            f"다음 후보 column을 찾지 못했습니다: {candidates}\n"
            f"실제 columns: {df.columns.tolist()}"
        )

    return None


PLAYER_ID_COL = resolve_col(
    tm_players_raw,
    ["player_id"],
)

PLAYER_NAME_COL = resolve_col(
    tm_players_raw,
    ["name", "player_name"],
)

PLAYER_DOB_COL = resolve_col(
    tm_players_raw,
    ["date_of_birth", "birth_date"],
    required=False,
)

PLAYER_COUNTRY_COL = resolve_col(
    tm_players_raw,
    [
        "country_of_citizenship",
        "nationality",
        "country",
    ],
    required=False,
)

VALUATION_PLAYER_ID_COL = resolve_col(
    valuations_raw,
    ["player_id"],
)

VALUATION_DATE_COL = resolve_col(
    valuations_raw,
    ["date", "datetime"],
)

VALUATION_VALUE_COL = resolve_col(
    valuations_raw,
    [
        "market_value_in_eur",
        "market_value",
    ],
)

TRANSFER_PLAYER_ID_COL = resolve_col(
    db_transfers_raw,
    ["player_id"],
)

TRANSFER_DATE_COL = resolve_col(
    db_transfers_raw,
    ["transfer_date", "date"],
)

TRANSFER_FROM_COL = resolve_col(
    db_transfers_raw,
    ["from_club_name", "from_club"],
)

TRANSFER_TO_COL = resolve_col(
    db_transfers_raw,
    ["to_club_name", "to_club"],
)

TRANSFER_FEE_COL = resolve_col(
    db_transfers_raw,
    ["transfer_fee", "fee"],
    required=False,
)

TRANSFER_MV_COL = resolve_col(
    db_transfers_raw,
    [
        "market_value_in_eur",
        "market_value",
    ],
    required=False,
)

GAME_COMP_COL = resolve_col(
    games_raw,
    ["competition_id"],
)

GAME_SEASON_COL = resolve_col(
    games_raw,
    ["season"],
)

GAME_DATE_COL = resolve_col(
    games_raw,
    ["date"],
)

print(
    "Resolved schema successfully."
)

Resolved schema successfully.


# Part B. Player ID Crosswalk

## 10. Transfermarkt 선수 master 생성

두 소스를 합쳐 `normalized name → player_id` 후보를 만듭니다.

- DuckDB `players`: 선수 master + 생년월일
- eordo ZIP: 이적 당시 이름 + player_id

### 자동 확정 원칙

**A 등급**
- 정규화 이름이 Transfermarkt 전체에서 단 하나의 player_id에만 연결됨

**B 등급**
- 동일 이름에 여러 player_id가 있으나 생년월일 기반 시즌 나이가 명확히 한 후보와 일치

**D 등급**
- fuzzy 후보
- 기본적으로 자동 확정하지 않고 audit만 생성

정확도를 coverage보다 우선합니다.

In [15]:
tm_players = pd.DataFrame({
    "player_id": pd.to_numeric(
        tm_players_raw[PLAYER_ID_COL],
        errors="coerce",
    ),
    "tm_player_name": (
        tm_players_raw[PLAYER_NAME_COL]
        .astype(str)
    ),
})

if PLAYER_DOB_COL is not None:
    tm_players["date_of_birth"] = pd.to_datetime(
        tm_players_raw[PLAYER_DOB_COL],
        errors="coerce",
    )
else:
    tm_players["date_of_birth"] = pd.NaT

if PLAYER_COUNTRY_COL is not None:
    tm_players["country_of_citizenship"] = (
        tm_players_raw[PLAYER_COUNTRY_COL]
        .astype(str)
    )
else:
    tm_players["country_of_citizenship"] = np.nan

tm_players = tm_players.dropna(
    subset=["player_id"]
).copy()

tm_players["player_id"] = (
    tm_players["player_id"]
    .astype(int)
)

tm_players["norm_player"] = (
    tm_players["tm_player_name"]
    .map(normalize_text)
)

eordo_name_ids = (
    eordo[
        [
            "player_id",
            "player_name",
            "norm_player",
        ]
    ]
    .dropna(subset=["player_id"])
    .drop_duplicates()
    .rename(
        columns={
            "player_name": "tm_player_name"
        }
    )
)

eordo_name_ids["player_id"] = (
    eordo_name_ids["player_id"]
    .astype(int)
)

duckdb_name_ids = (
    tm_players[
        [
            "player_id",
            "tm_player_name",
            "norm_player",
        ]
    ]
    .drop_duplicates()
)

name_id_master = pd.concat(
    [
        duckdb_name_ids,
        eordo_name_ids,
    ],
    ignore_index=True,
).drop_duplicates()

name_id_counts = (
    name_id_master
    .groupby("norm_player")
    ["player_id"]
    .nunique()
    .rename("candidate_id_count")
)

print(
    "Normalized Transfermarkt names:",
    name_id_counts.shape[0],
)

print(
    "Ambiguous normalized names:",
    int(
        (name_id_counts > 1).sum()
    ),
)

Normalized Transfermarkt names: 57713
Ambiguous normalized names: 1249


## 11. A 등급: Exact normalized name + unique player_id

In [16]:
player_crosswalk = (
    dev[
        ["player", "norm_player"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_ids_by_name = (
    name_id_master
    .groupby("norm_player")
    ["player_id"]
    .agg(
        lambda s: sorted(
            set(
                int(x)
                for x in s
                if pd.notna(x)
            )
        )
    )
)

player_crosswalk[
    "candidate_ids"
] = (
    player_crosswalk[
        "norm_player"
    ]
    .map(candidate_ids_by_name)
)

player_crosswalk[
    "candidate_ids"
] = (
    player_crosswalk[
        "candidate_ids"
    ]
    .apply(
        lambda x: (
            x
            if isinstance(x, list)
            else []
        )
    )
)

player_crosswalk[
    "candidate_id_count"
] = (
    player_crosswalk[
        "candidate_ids"
    ].map(len)
)

player_crosswalk[
    "tm_player_id"
] = np.nan

player_crosswalk[
    "match_method"
] = "unmatched"

player_crosswalk[
    "match_confidence"
] = "UNMATCHED"

mask_a = (
    player_crosswalk[
        "candidate_id_count"
    ]
    .eq(1)
)

player_crosswalk.loc[
    mask_a,
    "tm_player_id",
] = (
    player_crosswalk.loc[
        mask_a,
        "candidate_ids",
    ]
    .map(lambda x: x[0])
)

player_crosswalk.loc[
    mask_a,
    "match_method",
] = "exact_normalized_unique"

player_crosswalk.loc[
    mask_a,
    "match_confidence",
] = "A"

print(
    "A-grade players:",
    int(mask_a.sum()),
    "/",
    len(player_crosswalk),
    f"({mask_a.mean():.2%})",
)

A-grade players: 5584 / 6397 (87.29%)


## 12. B 등급: 동일 이름 여러 선수 → 생년월일/나이로 해결

예를 들어 `Fernando`, `Eduardo`처럼 동일 이름을 가진 여러 player_id가 존재할 수 있습니다.

각 후보의 생년월일로 현재 시즌 7월 1일 기준 예상 나이를 계산하고,
우리 데이터의 여러 시즌 나이와 가장 일관되게 맞는 ID를 찾습니다.

### B 등급 자동 확정 조건

- best 후보 median age error ≤ 1년
- 두 번째 후보보다 충분히 명확하게 좋음

조건이 애매하면 자동 확정하지 않습니다.

In [17]:
dob_by_id = (
    tm_players
    .drop_duplicates(
        subset=["player_id"]
    )
    .set_index("player_id")
    ["date_of_birth"]
    .to_dict()
)


def expected_age_on_july1(
    season_start,
    dob,
):
    if pd.isna(dob):
        return np.nan

    ref = pd.Timestamp(
        year=int(season_start),
        month=7,
        day=1,
    )

    return (
        (ref - pd.Timestamp(dob)).days
        / 365.2425
    )


def resolve_ambiguous_by_age(
    norm_player,
    candidate_ids,
):
    obs = dev.loc[
        dev["norm_player"].eq(
            norm_player
        ),
        ["season_start", "age"],
    ].dropna()

    if obs.empty:
        return None

    scores = []

    for pid in candidate_ids:
        dob = dob_by_id.get(
            int(pid),
            pd.NaT,
        )

        if pd.isna(dob):
            continue

        diffs = []

        for row in obs.itertuples(
            index=False
        ):
            exp_age = expected_age_on_july1(
                row.season_start,
                dob,
            )

            if pd.isna(exp_age):
                continue

            diffs.append(
                abs(
                    float(row.age)
                    - exp_age
                )
            )

        if diffs:
            scores.append({
                "player_id": int(pid),
                "median_age_error": float(
                    np.median(diffs)
                ),
                "mean_age_error": float(
                    np.mean(diffs)
                ),
                "n_age_rows": len(diffs),
            })

    if not scores:
        return None

    scores = sorted(
        scores,
        key=lambda x: (
            x["median_age_error"],
            x["mean_age_error"],
            -x["n_age_rows"],
        ),
    )

    best = scores[0]

    second_error = (
        scores[1]["median_age_error"]
        if len(scores) > 1
        else np.inf
    )

    # 나이 기준이 충분히 맞고
    # 2위와 구분될 때만 자동 확정
    if (
        best["median_age_error"] <= 1.0
        and (
            second_error
            - best["median_age_error"]
        ) >= 0.5
    ):
        return best

    return None


ambiguous_mask = (
    player_crosswalk[
        "candidate_id_count"
    ] > 1
)

resolved_b = 0

for idx in player_crosswalk.loc[
    ambiguous_mask
].index:

    norm_name = player_crosswalk.at[
        idx,
        "norm_player",
    ]

    candidates = player_crosswalk.at[
        idx,
        "candidate_ids",
    ]

    result = resolve_ambiguous_by_age(
        norm_name,
        candidates,
    )

    if result is None:
        continue

    player_crosswalk.at[
        idx,
        "tm_player_id",
    ] = result["player_id"]

    player_crosswalk.at[
        idx,
        "match_method",
    ] = "exact_name_age_resolved"

    player_crosswalk.at[
        idx,
        "match_confidence",
    ] = "B"

    player_crosswalk.at[
        idx,
        "age_match_error",
    ] = result[
        "median_age_error"
    ]

    resolved_b += 1


print(
    "B-grade resolved:",
    resolved_b,
)

B-grade resolved: 188


## 13. D 등급 fuzzy 후보 생성 — Audit Only

Exact normalized name으로 연결되지 않은 선수에 대해서는
가장 비슷한 Transfermarkt 이름 3개를 저장합니다.

기본 설정에서는 **fuzzy 후보를 자동으로 player_id에 넣지 않습니다.**

이 파일을 보고 정말 필요한 선수를 수동 검토한 뒤
`MANUAL_PLAYER_ID_OVERRIDES`에 추가하는 방식이 안전합니다.

In [18]:
tm_norm_names = sorted(
    name_id_master[
        "norm_player"
    ]
    .dropna()
    .unique()
)

unmatched_names = (
    player_crosswalk.loc[
        player_crosswalk[
            "tm_player_id"
        ].isna(),
        [
            "player",
            "norm_player",
        ],
    ]
    .drop_duplicates()
)

try:
    from rapidfuzz import process, fuzz

    FUZZY_ENGINE = "rapidfuzz"

    fuzzy_rows = []

    for row in unmatched_names.itertuples(
        index=False
    ):
        matches = process.extract(
            row.norm_player,
            tm_norm_names,
            scorer=fuzz.WRatio,
            limit=3,
        )

        for rank, (
            candidate_name,
            score,
            _,
        ) in enumerate(
            matches,
            start=1,
        ):
            ids = (
                name_id_master.loc[
                    name_id_master[
                        "norm_player"
                    ].eq(candidate_name),
                    "player_id",
                ]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

            fuzzy_rows.append({
                "player": row.player,
                "norm_player": row.norm_player,
                "rank": rank,
                "candidate_norm_name": candidate_name,
                "similarity": score / 100.0,
                "candidate_player_ids": ids,
            })

except ImportError:
    import difflib

    FUZZY_ENGINE = "difflib"

    fuzzy_rows = []

    for row in unmatched_names.itertuples(
        index=False
    ):
        matches = difflib.get_close_matches(
            row.norm_player,
            tm_norm_names,
            n=3,
            cutoff=0.70,
        )

        for rank, candidate_name in enumerate(
            matches,
            start=1,
        ):
            score = string_similarity(
                row.norm_player,
                candidate_name,
            )

            ids = (
                name_id_master.loc[
                    name_id_master[
                        "norm_player"
                    ].eq(candidate_name),
                    "player_id",
                ]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

            fuzzy_rows.append({
                "player": row.player,
                "norm_player": row.norm_player,
                "rank": rank,
                "candidate_norm_name": candidate_name,
                "similarity": score,
                "candidate_player_ids": ids,
            })


player_match_audit = pd.DataFrame(
    fuzzy_rows
)

print(
    "Fuzzy engine:",
    FUZZY_ENGINE,
)

print(
    "Unmatched unique players:",
    len(unmatched_names),
)

display(
    player_match_audit.head(20)
)

Fuzzy engine: difflib
Unmatched unique players: 625


,player,norm_player,rank,candidate_norm_name,similarity,candidate_player_ids
0,Abder Ramdane,abder ramdane,1,ylber ramadani,0.740741,[442703]
1,Abder Ramdane,abder ramdane,2,abderrahmane sarr,0.733333,[1144153]
2,Abder Ramdane,abder ramdane,3,abde raihani,0.720000,[860062]
3,Adaílton,adailton,1,adailton,1.000000,"[101253, 21853, 18699, 34371]"
4,Adaílton,adailton,2,ailton,0.857143,"[32860, 283863, 353214, 516]"
5,Adaílton,adailton,3,mailton,0.800000,[607210]
6,Adel Sellimi,adel sellimi,1,adul seidi,0.727273,[316828]
7,Aílton Gonçalves,ailton goncalves,1,anthony goncalves,0.848485,[111528]
8,Aílton Gonçalves,ailton goncalves,2,vitor goncalves,0.838710,[194547]
9,Aílton Gonçalves,ailton goncalves,3,ivo goncalves,0.827586,[45243]


## 14. 수동 player_id override 적용

Fuzzy audit를 검토한 뒤 확실한 선수만 위의
`MANUAL_PLAYER_ID_OVERRIDES`에 추가하면 됩니다.

Notebook을 다시 실행하면 자동 적용됩니다.

In [19]:
override_map = {
    normalize_text(name): int(pid)
    for name, pid
    in MANUAL_PLAYER_ID_OVERRIDES.items()
}

manual_mask = (
    player_crosswalk[
        "norm_player"
    ].isin(
        override_map.keys()
    )
)

if manual_mask.any():
    player_crosswalk.loc[
        manual_mask,
        "tm_player_id",
    ] = (
        player_crosswalk.loc[
            manual_mask,
            "norm_player",
        ]
        .map(override_map)
    )

    player_crosswalk.loc[
        manual_mask,
        "match_method",
    ] = "manual_override"

    player_crosswalk.loc[
        manual_mask,
        "match_confidence",
    ] = "MANUAL"


player_crosswalk[
    "tm_player_id"
] = (
    pd.to_numeric(
        player_crosswalk[
            "tm_player_id"
        ],
        errors="coerce",
    )
    .astype("Int64")
)

display(
    player_crosswalk[
        "match_confidence"
    ]
    .value_counts(
        dropna=False
    )
    .rename("n")
    .to_frame()
)

,n
match_confidence,
A,5584
UNMATCHED,625
B,188


## 15. 선수 ID를 모든 player-season 행에 연결

In [20]:
snapshot = dev.merge(
    player_crosswalk[
        [
            "player",
            "norm_player",
            "tm_player_id",
            "match_method",
            "match_confidence",
        ]
    ],
    on=[
        "player",
        "norm_player",
    ],
    how="left",
    validate="many_to_one",
)

player_row_match_rate = (
    snapshot[
        "tm_player_id"
    ]
    .notna()
    .mean()
)

player_unique_match_rate = (
    player_crosswalk[
        "tm_player_id"
    ]
    .notna()
    .mean()
)

print(
    f"Unique player ID match: "
    f"{player_unique_match_rate:.2%}"
)

print(
    f"Player-season row match: "
    f"{player_row_match_rate:.2%}"
)

assert (
    player_row_match_rate
    >= MIN_PLAYER_ROW_MATCH_RATE
), (
    "Player ID row 매칭률이 예상보다 낮습니다. "
    "crosswalk audit를 먼저 확인하세요."
)

player_season_crosswalk = (
    snapshot[
        [
            "row_id",
            "player",
            "season",
            "team",
            "age",
            "tm_player_id",
            "match_method",
            "match_confidence",
        ]
    ]
    .copy()
)

unmatched_players = (
    player_crosswalk.loc[
        player_crosswalk[
            "tm_player_id"
        ].isna()
    ]
    .copy()
)

display(
    unmatched_players.head(20)
)

Unique player ID match: 90.23%
Player-season row match: 92.57%


,player,norm_player,candidate_ids,candidate_id_count,tm_player_id,match_method,match_confidence,age_match_error
1,Abder Ramdane,abder ramdane,[],0,<NA>,unmatched,UNMATCHED,NaN
2,Adaílton,adailton,"[18699, 21853, 34371, 101253]",4,<NA>,unmatched,UNMATCHED,NaN
4,Adel Sellimi,adel sellimi,[],0,<NA>,unmatched,UNMATCHED,NaN
8,Aílton Gonçalves,ailton goncalves,[],0,<NA>,unmatched,UNMATCHED,NaN
17,Aleksandr Mostovoi,aleksandr mostovoi,[],0,<NA>,unmatched,UNMATCHED,NaN
18,Aleksandre Iashvili,aleksandre iashvili,[],0,<NA>,unmatched,UNMATCHED,NaN
22,Alessandro Mazzola,alessandro mazzola,[],0,<NA>,unmatched,UNMATCHED,NaN
26,Alex Fernández,alex fernandez,"[7885, 89733]",2,<NA>,unmatched,UNMATCHED,NaN
30,Alexander Strehmel,alexander strehmel,[],0,<NA>,unmatched,UNMATCHED,NaN
32,Alexey Smertin,alexey smertin,[],0,<NA>,unmatched,UNMATCHED,NaN


# Part C. Prediction Cutoff

## 16. 시즌별 cutoff 생성

### 기본 정책

가능한 시즌:
- DuckDB `games`에서 Big 5 리그 각각의 첫 경기 날짜를 찾음
- **Big 5 전체 중 가장 빠른 개막일의 전날**을 cutoff로 사용

왜 한 리그별 cutoff가 아니라 Big 5 공통 cutoff인가?

예를 들어 EPL이 이미 시작된 뒤 Bundesliga 개막 직전 정보를 사용하면
Bundesliga 선수는 더 많은 미래 정보를 갖게 됩니다.

따라서 같은 시즌 모든 선수에게 하나의 timestamp를 적용합니다.

### 과거 경기 일정이 DuckDB에 없는 시즌

`YYYY-07-31`을 conservative fallback으로 사용하고
`cutoff_source = fallback_jul31`로 명확히 표시합니다.

08-02에서 필요하면 fallback 시즌만 제외한 sensitivity analysis도 할 수 있습니다.

In [21]:
games = games_raw.copy()

games["competition_id"] = (
    games[GAME_COMP_COL]
    .astype(str)
)

games["season_start"] = (
    pd.to_numeric(
        games[GAME_SEASON_COL],
        errors="coerce",
    )
)

games["game_date"] = (
    pd.to_datetime(
        games[GAME_DATE_COL],
        errors="coerce",
    )
)

big5_game_rows = (
    games[
        games["competition_id"]
        .isin(
            BIG5_COMPETITION_IDS.values()
        )
    ]
    .dropna(
        subset=[
            "season_start",
            "game_date",
        ]
    )
    .copy()
)

big5_game_rows[
    "season_start"
] = (
    big5_game_rows[
        "season_start"
    ]
    .astype(int)
)

first_big5_game_by_year = (
    big5_game_rows
    .groupby("season_start")
    ["game_date"]
    .min()
    .to_dict()
)

cutoff_rows = []

for target_year in sorted(
    snapshot["target_year"]
    .dropna()
    .astype(int)
    .unique()
):
    if (
        target_year
        in first_big5_game_by_year
    ):
        first_game = (
            first_big5_game_by_year[
                target_year
            ]
        )

        cutoff_date = (
            first_game
            - pd.Timedelta(days=1)
        )

        source = "duckdb_big5_first_game"

    else:
        first_game = pd.NaT

        cutoff_date = pd.Timestamp(
            year=int(target_year),
            month=FALLBACK_CUTOFF_MONTH,
            day=FALLBACK_CUTOFF_DAY,
        )

        source = "fallback_jul31"

    cutoff_rows.append({
        "target_year": int(target_year),
        "first_big5_game_date": first_game,
        "cutoff_date": cutoff_date,
        "cutoff_source": source,
    })


cutoff_table = pd.DataFrame(
    cutoff_rows
)

snapshot = snapshot.merge(
    cutoff_table,
    on="target_year",
    how="left",
    validate="many_to_one",
)

display(cutoff_table)

print("\nCutoff source counts")
display(
    cutoff_table[
        "cutoff_source"
    ]
    .value_counts()
    .rename("seasons")
    .to_frame()
)

,target_year,first_big5_game_date,cutoff_date,cutoff_source
0,2001,NaT,2001-07-31,fallback_jul31
1,2002,NaT,2002-07-31,fallback_jul31
2,2003,NaT,2003-07-31,fallback_jul31
3,2004,NaT,2004-07-31,fallback_jul31
4,2005,NaT,2005-07-31,fallback_jul31
5,2006,NaT,2006-07-31,fallback_jul31
6,2007,NaT,2007-07-31,fallback_jul31
7,2008,NaT,2008-07-31,fallback_jul31
8,2009,NaT,2009-07-31,fallback_jul31
9,2010,NaT,2010-07-31,fallback_jul31



Cutoff source counts


,seasons
cutoff_source,
duckdb_big5_first_game,14
fallback_jul31,11


# Part D. Summer Transfer Event 정리

## 17. eordo summer in/out → canonical transfer event

v2에서도 eordo 데이터는 다음 정보 때문에 유지합니다.

- summer / winter
- in / out
- fee
- is_loan
- transfer 당시 market value
- Big 5 쪽 league / country

다만 **날짜와 이동 순서의 기준은 DuckDB transfer history를 함께 사용**합니다.

`Laliga` 같은 리그 이름은 이미 `La Liga`로 canonicalize된 상태입니다.

In [22]:
summer = (
    eordo[
        eordo["window"]
        .astype(str)
        .str.lower()
        .eq("summer")
    ]
    .copy()
)

summer["movement_lower"] = (
    summer["movement"]
    .astype(str)
    .str.lower()
)

is_in = (
    summer["movement_lower"]
    .eq("in")
)

summer["from_club"] = np.where(
    is_in,
    summer["dealing_club"],
    summer["club"],
)

summer["to_club"] = np.where(
    is_in,
    summer["club"],
    summer["dealing_club"],
)

summer["from_league"] = np.where(
    is_in,
    np.nan,
    summer["league"],
)

summer["to_league"] = np.where(
    is_in,
    summer["league"],
    np.nan,
)

summer["from_country"] = np.where(
    is_in,
    summer["dealing_country"],
    summer["league"].map(
        LEAGUE_COUNTRY
    ),
)

summer["to_country"] = np.where(
    is_in,
    summer["league"].map(
        LEAGUE_COUNTRY
    ),
    summer["dealing_country"],
)

summer["from_league"] = (
    pd.Series(
        summer["from_league"],
        index=summer.index,
    )
    .map(canonicalize_league)
)

summer["to_league"] = (
    pd.Series(
        summer["to_league"],
        index=summer.index,
    )
    .map(canonicalize_league)
)

summer["norm_from_club"] = (
    summer["from_club"]
    .map(normalize_club)
)

summer["norm_to_club"] = (
    summer["to_club"]
    .map(normalize_club)
)

summer["market_value"] = pd.to_numeric(
    summer["market_value"],
    errors="coerce",
)

summer["fee"] = pd.to_numeric(
    summer["fee"],
    errors="coerce",
)

summer["is_loan"] = (
    pd.to_numeric(
        summer["is_loan"],
        errors="coerce",
    )
    .fillna(0)
    .astype(int)
)


def first_nonnull(series):
    series = series.dropna()

    if series.empty:
        return np.nan

    return series.iloc[0]


transfer_events = (
    summer
    .groupby(
        [
            "player_id",
            "season",
            "norm_from_club",
            "norm_to_club",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        player_name=(
            "player_name",
            first_nonnull,
        ),
        from_club=(
            "from_club",
            first_nonnull,
        ),
        to_club=(
            "to_club",
            first_nonnull,
        ),
        from_league=(
            "from_league",
            first_nonnull,
        ),
        to_league=(
            "to_league",
            first_nonnull,
        ),
        from_country=(
            "from_country",
            first_nonnull,
        ),
        to_country=(
            "to_country",
            first_nonnull,
        ),
        transfer_market_value_eur=(
            "market_value",
            "max",
        ),
        transfer_fee_eur=(
            "fee",
            "max",
        ),
        is_loan=(
            "is_loan",
            "max",
        ),
        has_big5_in_record=(
            "movement_lower",
            lambda s: int(
                (s == "in").any()
            ),
        ),
        has_big5_out_record=(
            "movement_lower",
            lambda s: int(
                (s == "out").any()
            ),
        ),
        source_rows=(
            "movement_lower",
            "size",
        ),
    )
)

transfer_events[
    "player_id"
] = (
    transfer_events[
        "player_id"
    ]
    .astype(int)
)

transfer_events[
    "season"
] = (
    transfer_events[
        "season"
    ]
    .astype(int)
)

transfer_events[
    "event_id"
] = np.arange(
    len(transfer_events),
    dtype=int,
)

print(
    "Canonical summer events:",
    len(transfer_events),
)

print(
    "Events represented by both in/out rows:",
    int(
        (
            transfer_events[
                "source_rows"
            ] > 1
        ).sum()
    ),
)

display(
    transfer_events.head()
)

Canonical summer events: 50672
Events represented by both in/out rows: 10071


,player_id,season,norm_from_club,norm_to_club,player_name,from_club,to_club,from_league,to_league,from_country,to_country,transfer_market_value_eur,transfer_fee_eur,is_loan,has_big5_in_record,has_big5_out_record,source_rows,event_id
0,1,2003,1 fc kaiserslautern,vfb lubeck,Silvio Adzic,1.FC Kaiserslautern,VfB Lübeck,Bundesliga,NaN,Germany,Germany,NaN,0.0,0,0,1,1,0
1,2,2003,1 fc kaiserslautern,al rayyan sc,Mario Basler,1.FC Kaiserslautern,Al-Rayyan SC,Bundesliga,NaN,Germany,Qatar,NaN,0.0,0,0,1,1,1
2,4,2004,bolton wanderers,without club,Youri Djorkaeff,Bolton Wanderers,Without Club,Premier League,NaN,England,NaN,NaN,NaN,0,0,1,1,2
3,4,2004,without club,blackburn rovers,Youri Djorkaeff,Without Club,Blackburn Rovers,NaN,Premier League,NaN,England,NaN,NaN,0,1,0,1,3
4,5,2000,ac sparta prague,1 fc kaiserslautern,Petr Gabriel,AC Sparta Prague,1.FC Kaiserslautern,NaN,Bundesliga,Czech Republic,Germany,NaN,1500000.0,0,1,0,1,4


## 18. v2 — eordo event와 DuckDB transfer date를 **1:1 assignment**로 연결

v1은 각 eordo event가 독립적으로 DuckDB 후보 하나를 고르는 방식이었습니다.

한 선수에게 같은 여름 여러 이벤트가 있으면:

```text
이벤트 A ─┐
          ├→ 같은 DB 후보를 서로 선택
이벤트 B ─┘
```

처럼 모호해질 수 있었습니다.

v2에서는 같은 `(player_id, target_year)` 안에서 모든 pair score를 계산한 뒤
**각 eordo event와 각 DuckDB transfer row가 최대 한 번씩만 사용되는 1:1 greedy assignment**를 합니다.

Club 비교도 `FC / CF / Olympique / Borussia` 같은 표기에 더 강한
`club_similarity()`를 사용합니다.

In [23]:
db_transfers = pd.DataFrame({
    "player_id": pd.to_numeric(
        db_transfers_raw[
            TRANSFER_PLAYER_ID_COL
        ],
        errors="coerce",
    ),
    "transfer_date": pd.to_datetime(
        db_transfers_raw[
            TRANSFER_DATE_COL
        ],
        errors="coerce",
    ),
    "db_from_club": (
        db_transfers_raw[
            TRANSFER_FROM_COL
        ]
        .astype(str)
    ),
    "db_to_club": (
        db_transfers_raw[
            TRANSFER_TO_COL
        ]
        .astype(str)
    ),
})

if TRANSFER_FEE_COL is not None:
    db_transfers[
        "db_transfer_fee_eur"
    ] = pd.to_numeric(
        db_transfers_raw[
            TRANSFER_FEE_COL
        ],
        errors="coerce",
    )
else:
    db_transfers[
        "db_transfer_fee_eur"
    ] = np.nan

if TRANSFER_MV_COL is not None:
    db_transfers[
        "db_market_value_eur"
    ] = pd.to_numeric(
        db_transfers_raw[
            TRANSFER_MV_COL
        ],
        errors="coerce",
    )
else:
    db_transfers[
        "db_market_value_eur"
    ] = np.nan


db_transfers = (
    db_transfers
    .dropna(
        subset=[
            "player_id",
            "transfer_date",
        ]
    )
    .copy()
)

db_transfers[
    "player_id"
] = (
    db_transfers[
        "player_id"
    ]
    .astype(int)
)

db_transfers[
    "transfer_year"
] = (
    db_transfers[
        "transfer_date"
    ]
    .dt.year
)

db_transfers[
    "norm_db_from"
] = (
    db_transfers[
        "db_from_club"
    ]
    .map(normalize_club)
)

db_transfers[
    "norm_db_to"
] = (
    db_transfers[
        "db_to_club"
    ]
    .map(normalize_club)
)

db_transfers[
    "db_row_id"
] = np.arange(
    len(db_transfers),
    dtype=int,
)

db_transfer_groups = {
    key: group.reset_index(
        drop=True
    )
    for key, group
    in db_transfers.groupby(
        [
            "player_id",
            "transfer_year",
        ]
    )
}


def transfer_pair_score(
    e_from,
    e_to,
    d_from,
    d_to,
):
    from_score = club_similarity(
        e_from,
        d_from,
    )

    to_score = club_similarity(
        e_to,
        d_to,
    )

    # 두 방향 모두 중요하므로 평균 사용
    return (
        from_score + to_score
    ) / 2.0


date_match_rows = []

# 같은 player-year 안에서 event들을 동시에 배정
for (
    player_id,
    season,
), ev_group in transfer_events.groupby(
    ["player_id", "season"]
):
    ev_group = ev_group.copy()

    db_group = db_transfer_groups.get(
        (
            int(player_id),
            int(season),
        )
    )

    # 기본 no-candidate 레코드
    if (
        db_group is None
        or db_group.empty
    ):
        for ev in ev_group.itertuples():
            date_match_rows.append({
                "event_id": ev.event_id,
                "player_id": ev.player_id,
                "season": ev.season,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "transfer_date": pd.NaT,
                "db_match_score": np.nan,
                "db_date_match_method": "no_db_candidate",
                "db_transfer_fee_eur": np.nan,
                "db_market_value_eur": np.nan,
            })
        continue

    pair_candidates = []

    for ev in ev_group.itertuples():
        for db_row in db_group.itertuples():
            from_score = club_similarity(
                ev.from_club,
                db_row.db_from_club,
            )

            to_score = club_similarity(
                ev.to_club,
                db_row.db_to_club,
            )

            pair_score = (
                from_score + to_score
            ) / 2.0

            pair_candidates.append({
                "event_id": int(ev.event_id),
                "db_row_id": int(db_row.db_row_id),
                "pair_score": float(pair_score),
                "from_score": float(from_score),
                "to_score": float(to_score),
            })

    pair_candidates = sorted(
        pair_candidates,
        key=lambda x: (
            x["pair_score"],
            min(
                x["from_score"],
                x["to_score"],
            ),
        ),
        reverse=True,
    )

    assigned_events = set()
    assigned_db_rows = set()
    assignments = {}

    for pair in pair_candidates:
        event_id = pair["event_id"]
        db_row_id = pair["db_row_id"]

        if event_id in assigned_events:
            continue

        if db_row_id in assigned_db_rows:
            continue

        strong_direction = (
            pair["from_score"] >= 0.82
            or pair["to_score"] >= 0.82
        )

        acceptable = (
            pair["pair_score"]
            >= TRANSFER_PAIR_STRONG_SCORE
            or (
                pair["pair_score"]
                >= TRANSFER_PAIR_MIN_SCORE
                and strong_direction
            )
        )

        if not acceptable:
            continue

        assignments[event_id] = pair

        assigned_events.add(
            event_id
        )

        assigned_db_rows.add(
            db_row_id
        )

    for ev in ev_group.itertuples():
        base = {
            "event_id": ev.event_id,
            "player_id": ev.player_id,
            "season": ev.season,
            "from_club": ev.from_club,
            "to_club": ev.to_club,
        }

        pair = assignments.get(
            int(ev.event_id)
        )

        if pair is None:
            best_scores = [
                p
                for p in pair_candidates
                if p["event_id"]
                == int(ev.event_id)
            ]

            best_score = (
                best_scores[0]["pair_score"]
                if best_scores
                else np.nan
            )

            date_match_rows.append({
                **base,
                "transfer_date": pd.NaT,
                "db_match_score": best_score,
                "db_date_match_method": "unresolved_after_1to1_assignment",
                "db_transfer_fee_eur": np.nan,
                "db_market_value_eur": np.nan,
            })

            continue

        db_match = db_group.loc[
            db_group[
                "db_row_id"
            ].eq(
                pair["db_row_id"]
            )
        ].iloc[0]

        exact_pair = (
            normalize_club(
                ev.from_club
            )
            == normalize_club(
                db_match[
                    "db_from_club"
                ]
            )
            and normalize_club(
                ev.to_club
            )
            == normalize_club(
                db_match[
                    "db_to_club"
                ]
            )
        )

        if exact_pair:
            method = (
                "exact_player_year_club_pair"
            )
        elif (
            pair["pair_score"]
            >= TRANSFER_PAIR_STRONG_SCORE
        ):
            method = (
                "strong_1to1_club_pair"
            )
        else:
            method = (
                "directional_1to1_club_pair"
            )

        date_match_rows.append({
            **base,
            "transfer_date": db_match[
                "transfer_date"
            ],
            "db_match_score": pair[
                "pair_score"
            ],
            "db_date_match_method": method,
            "db_transfer_fee_eur": db_match[
                "db_transfer_fee_eur"
            ],
            "db_market_value_eur": db_match[
                "db_market_value_eur"
            ],
        })


transfer_date_matches = pd.DataFrame(
    date_match_rows
)

transfer_events = (
    transfer_events
    .merge(
        transfer_date_matches[
            [
                "event_id",
                "transfer_date",
                "db_match_score",
                "db_date_match_method",
                "db_transfer_fee_eur",
                "db_market_value_eur",
            ]
        ],
        on="event_id",
        how="left",
        validate="one_to_one",
    )
)

# eordo 값을 우선 사용하고 없을 때 DB 값 fallback
transfer_events[
    "transfer_fee_eur"
] = (
    transfer_events[
        "transfer_fee_eur"
    ]
    .combine_first(
        transfer_events[
            "db_transfer_fee_eur"
        ]
    )
)

transfer_events[
    "transfer_market_value_eur"
] = (
    transfer_events[
        "transfer_market_value_eur"
    ]
    .combine_first(
        transfer_events[
            "db_market_value_eur"
        ]
    )
)

transfer_date_match_rate = (
    transfer_events[
        "transfer_date"
    ]
    .notna()
    .mean()
)

transfer_date_coverage_by_year = (
    transfer_events
    .assign(
        date_known=lambda d:
            d["transfer_date"].notna()
    )
    .groupby("season")
    .agg(
        events=("event_id", "size"),
        dated_events=(
            "date_known",
            "sum",
        ),
        date_coverage=(
            "date_known",
            "mean",
        ),
    )
    .reset_index()
)

print(
    "Summer event exact-date coverage:",
    f"{transfer_date_match_rate:.2%}"
)

display(
    transfer_events[
        "db_date_match_method"
    ]
    .value_counts(
        dropna=False
    )
    .rename("n")
    .to_frame()
)

display(
    transfer_date_coverage_by_year.tail(15)
)

Summer event exact-date coverage: 27.86%


,n
db_date_match_method,
no_db_candidate,36205
strong_1to1_club_pair,11938
exact_player_year_club_pair,1483
directional_1to1_club_pair,694
unresolved_after_1to1_assignment,352


,season,events,dated_events,date_coverage
11,2011,2089,194,0.092867
12,2012,2092,231,0.110421
13,2013,2274,340,0.149516
14,2014,2405,472,0.196258
15,2015,2302,615,0.267159
16,2016,2227,686,0.308038
17,2017,2149,792,0.368544
18,2018,2133,919,0.430849
19,2019,2225,1069,0.480449
20,2020,1919,1022,0.532569


## 19. v2 — 한 선수의 summer transfer를 **시간순 team-state transition**으로 추적

v1에서는 player-season마다 transfer event 하나만 골랐습니다.

하지만 실제 이적은 한 여름에 여러 이벤트가 연속될 수 있습니다.

예: Guirassy 2023

```text
관측 현재 팀: Stuttgart

06-30 Stuttgart → Rennes   (임대 복귀)
07-01 Rennes → Stuttgart   (완전 이적)

최종 preseason 팀: Stuttgart
```

v1 방식은 이벤트 하나만 잡으면서 Rennes로 잘못 이동시킬 수 있었습니다.

v2는:

1. 현재 팀에서 시작
2. cutoff 이전 dated event를 날짜순 정렬
3. 현재 team state와 `from_club`이 맞으면 실제 이동 적용
4. `to_club`이 현재 team state와 맞으면 계약 전환/입단 처리로 보고 팀은 유지
5. 다음 event에서 갱신된 team state를 다시 사용
6. cutoff 시점의 최종 team state를 destination으로 사용

합니다.

In [24]:
events_by_player_year = {
    key: group.sort_values(
        [
            "transfer_date",
            "event_id",
        ],
        na_position="last",
    ).reset_index(
        drop=True
    )
    for key, group
    in transfer_events.groupby(
        [
            "player_id",
            "season",
        ]
    )
}


def trace_preseason_transfer_timeline(
    current_team,
    current_league,
    player_id,
    target_year,
    cutoff_date,
):
    """
    cutoff 이전 dated event만 실제 feature에 반영.

    unknown-date / post-cutoff event는
    audit count로만 반환하고 model feature에는 쓰지 않음.
    """
    base_country = (
        LEAGUE_COUNTRY.get(
            current_league,
            np.nan,
        )
    )

    result = {
        "timeline_status": None,
        "confirmed_preseason_event_count": 0,
        "preseason_team_transition_count": 0,
        "selected_event_id": np.nan,
        "destination_team": current_team,
        "destination_league": current_league,
        "destination_country": base_country,
        "unknown_date_context_event_count_audit": 0,
        "post_cutoff_context_event_count_audit": 0,
        "context_mismatch_pre_event_count_audit": 0,
        "timeline_trace": "",
    }

    if pd.isna(player_id):
        result[
            "timeline_status"
        ] = "no_player_id"

        return result

    key = (
        int(player_id),
        int(target_year),
    )

    candidates = (
        events_by_player_year.get(
            key
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        result[
            "timeline_status"
        ] = "no_summer_event"

        return result

    cutoff_date = pd.Timestamp(
        cutoff_date
    )

    known = candidates.loc[
        candidates[
            "transfer_date"
        ].notna()
    ].copy()

    unknown = candidates.loc[
        candidates[
            "transfer_date"
        ].isna()
    ].copy()

    pre = known.loc[
        known[
            "transfer_date"
        ]
        <= cutoff_date
    ].sort_values(
        [
            "transfer_date",
            "event_id",
        ]
    )

    post = known.loc[
        known[
            "transfer_date"
        ]
        > cutoff_date
    ].sort_values(
        [
            "transfer_date",
            "event_id",
        ]
    )

    # unknown/post event는 audit만.
    # "미래에 event가 존재한다"는 사실 자체를
    # model feature로 넣으면 leakage가 될 수 있기 때문.
    for _, ev in unknown.iterrows():
        if max(
            club_similarity(
                current_team,
                ev["from_club"],
            ),
            club_similarity(
                current_team,
                ev["to_club"],
            ),
        ) >= TEAM_CONTEXT_MIN_SCORE:
            result[
                "unknown_date_context_event_count_audit"
            ] += 1

    for _, ev in post.iterrows():
        if max(
            club_similarity(
                current_team,
                ev["from_club"],
            ),
            club_similarity(
                current_team,
                ev["to_club"],
            ),
        ) >= TEAM_CONTEXT_MIN_SCORE:
            result[
                "post_cutoff_context_event_count_audit"
            ] += 1

    team_state = current_team
    league_state = current_league
    country_state = base_country

    trace_parts = []
    relevant_events = []

    for _, ev in pre.iterrows():
        from_score = club_similarity(
            team_state,
            ev["from_club"],
        )

        to_score = club_similarity(
            team_state,
            ev["to_club"],
        )

        # A) 현재 team state에서 나가는 event
        if (
            from_score
            >= TEAM_CONTEXT_MIN_SCORE
        ):
            old_state = team_state

            team_state = ev[
                "to_club"
            ]

            if pd.notna(
                ev["to_league"]
            ):
                league_state = (
                    canonicalize_league(
                        ev["to_league"]
                    )
                )
            else:
                # Big5 outbound인데 destination league가
                # 없는 경우는 Big5 밖으로 이동한 것으로 처리.
                league_state = np.nan

            if pd.notna(
                ev["to_country"]
            ):
                country_state = ev[
                    "to_country"
                ]
            elif (
                pd.notna(league_state)
                and league_state
                in LEAGUE_COUNTRY
            ):
                country_state = (
                    LEAGUE_COUNTRY[
                        league_state
                    ]
                )
            else:
                country_state = np.nan

            result[
                "confirmed_preseason_event_count"
            ] += 1

            # 최종적으로 같은 이름 변형이 아닌
            # 실질 팀 이동일 때 transition count 증가
            if (
                club_similarity(
                    old_state,
                    team_state,
                )
                < SAME_CLUB_SCORE
            ):
                result[
                    "preseason_team_transition_count"
                ] += 1

            relevant_events.append(
                ev
            )

            trace_parts.append(
                f"{ev['transfer_date'].date()}: "
                f"{old_state} -> {team_state}"
            )

            continue

        # B) 현재 team state로 들어오는 event
        #    예: 임대 복귀 직후 완전 이적.
        #    실제 경기 팀은 이미 현재 팀이므로 state 유지.
        if (
            to_score
            >= TEAM_CONTEXT_MIN_SCORE
        ):
            result[
                "confirmed_preseason_event_count"
            ] += 1

            relevant_events.append(
                ev
            )

            # league/country가 알려져 있으면
            # 현재 팀과 일관된 정보로 갱신 가능
            if pd.notna(
                ev["to_league"]
            ):
                league_state = (
                    canonicalize_league(
                        ev["to_league"]
                    )
                )

            if pd.notna(
                ev["to_country"]
            ):
                country_state = ev[
                    "to_country"
                ]

            trace_parts.append(
                f"{ev['transfer_date'].date()}: "
                f"arrival/contract -> {team_state}"
            )

            continue

        result[
            "context_mismatch_pre_event_count_audit"
        ] += 1

    if relevant_events:
        last_event = (
            relevant_events[-1]
        )

        result[
            "selected_event_id"
        ] = int(
            last_event[
                "event_id"
            ]
        )

        result[
            "timeline_status"
        ] = (
            "confirmed_pre_cutoff_event"
        )

    else:
        result[
            "timeline_status"
        ] = (
            "no_confirmed_pre_cutoff_event"
        )

    result[
        "destination_team"
    ] = team_state

    result[
        "destination_league"
    ] = league_state

    result[
        "destination_country"
    ] = country_state

    result[
        "timeline_trace"
    ] = " | ".join(
        trace_parts
    )

    return result


timeline_rows = []

for row in snapshot[
    [
        "row_id",
        "team",
        "league",
        "tm_player_id",
        "target_year",
        "cutoff_date",
    ]
].itertuples(
    index=False
):
    result = (
        trace_preseason_transfer_timeline(
            current_team=row.team,
            current_league=row.league,
            player_id=row.tm_player_id,
            target_year=row.target_year,
            cutoff_date=row.cutoff_date,
        )
    )

    timeline_rows.append({
        "row_id": row.row_id,
        **result,
    })


transfer_timeline = pd.DataFrame(
    timeline_rows
)

snapshot = snapshot.merge(
    transfer_timeline,
    on="row_id",
    how="left",
    validate="one_to_one",
)

# 최종 economics는 cutoff 이전 관련 event 중
# 마지막 event의 값을 사용
selected_event_features = (
    transfer_events[
        [
            "event_id",
            "transfer_date",
            "from_club",
            "to_club",
            "from_league",
            "to_league",
            "from_country",
            "to_country",
            "transfer_fee_eur",
            "transfer_market_value_eur",
            "is_loan",
            "db_date_match_method",
            "db_match_score",
        ]
    ]
    .add_prefix("tr_")
)

snapshot = snapshot.merge(
    selected_event_features,
    left_on="selected_event_id",
    right_on="tr_event_id",
    how="left",
    validate="many_to_one",
)


# --------------------------------------------------------------
# Audit 파일: model feature가 아님
# --------------------------------------------------------------

unknown_date_audit_rows = []
post_cutoff_audit_rows = []

for row in snapshot[
    [
        "row_id",
        "player",
        "season",
        "target_season",
        "team",
        "tm_player_id",
        "target_year",
        "cutoff_date",
    ]
].itertuples(
    index=False
):
    if pd.isna(
        row.tm_player_id
    ):
        continue

    candidates = (
        events_by_player_year.get(
            (
                int(row.tm_player_id),
                int(row.target_year),
            )
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        continue

    for ev in candidates.itertuples():
        context_score = max(
            club_similarity(
                row.team,
                ev.from_club,
            ),
            club_similarity(
                row.team,
                ev.to_club,
            ),
        )

        if (
            context_score
            < TEAM_CONTEXT_MIN_SCORE
        ):
            continue

        if pd.isna(
            ev.transfer_date
        ):
            unknown_date_audit_rows.append({
                "row_id": row.row_id,
                "player": row.player,
                "season": row.season,
                "target_season": row.target_season,
                "team": row.team,
                "cutoff_date": row.cutoff_date,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "context_score": context_score,
                "db_date_match_method": ev.db_date_match_method,
            })

        elif (
            pd.Timestamp(
                ev.transfer_date
            )
            > pd.Timestamp(
                row.cutoff_date
            )
        ):
            post_cutoff_audit_rows.append({
                "row_id": row.row_id,
                "player": row.player,
                "season": row.season,
                "target_season": row.target_season,
                "team": row.team,
                "cutoff_date": row.cutoff_date,
                "transfer_date": ev.transfer_date,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "context_score": context_score,
            })


unknown_date_transfer_audit = (
    pd.DataFrame(
        unknown_date_audit_rows
    )
)

post_cutoff_transfer_audit = (
    pd.DataFrame(
        post_cutoff_audit_rows
    )
)

print(
    "Rows with confirmed pre-cutoff event:",
    int(
        (
            snapshot[
                "confirmed_preseason_event_count"
            ] > 0
        ).sum()
    ),
)

print(
    "Rows whose final team changed:",
    int(
        (
            snapshot[
                "preseason_team_transition_count"
            ] > 0
        ).sum()
    ),
)

print(
    "Unknown-date context events (audit only):",
    len(
        unknown_date_transfer_audit
    ),
)

print(
    "Post-cutoff context events (audit only):",
    len(
        post_cutoff_transfer_audit
    ),
)

display(
    snapshot.loc[
        snapshot[
            "confirmed_preseason_event_count"
        ] > 0,
        [
            "player",
            "season",
            "team",
            "cutoff_date",
            "confirmed_preseason_event_count",
            "preseason_team_transition_count",
            "destination_team",
            "timeline_trace",
        ],
    ].head(20)
)

Rows with confirmed pre-cutoff event: 1465
Rows whose final team changed: 1444
Unknown-date context events (audit only): 4099
Post-cutoff context events (audit only): 626


,player,season,team,cutoff_date,confirmed_preseason_event_count,preseason_team_transition_count,destination_team,timeline_trace
3105,James Milner,2003-2004,Leeds United,2004-07-31,1,1,Newcastle United,2004-07-02: Leeds United -> Newcastle United
5152,Lukas Podolski,2005-2006,Köln,2006-07-31,1,1,Bayern Munich,2006-07-10: Köln -> Bayern Munich
5407,Santi Cazorla,2005-2006,Villarreal,2006-07-31,1,1,Recreativo Huelva,2006-07-07: Villarreal -> Recreativo Huelva
5579,André-Pierre Gignac,2006-2007,Lorient,2007-07-31,1,1,FC Toulouse,2007-07-01: Lorient -> FC Toulouse
6326,Raúl García,2006-2007,Osasuna,2007-07-31,1,1,Atlético de Madrid,2007-07-01: Osasuna -> Atlético de Madrid
6375,Santi Cazorla,2006-2007,Recreativo,2007-07-31,1,1,Villarreal CF,2007-07-01: Recreativo -> Villarreal CF
6430,Steven Davis,2006-2007,Aston Villa,2007-07-31,1,1,Fulham FC,2007-07-01: Aston Villa -> Fulham FC
7073,Kévin Gameiro,2007-2008,Strasbourg,2008-07-31,1,1,FC Lorient,2008-07-01: Strasbourg -> FC Lorient
7231,Miralem Pjanić,2007-2008,Metz,2008-07-31,1,1,Olympique Lyon,2008-07-01: Metz -> Olympique Lyon
7554,Álvaro Negredo,2008-2009,Almería,2009-07-31,1,1,Real Madrid,2009-07-01: Almería -> Real Madrid


## 20. v2 — Leakage-safe Transfer Feature 생성

### 의미를 명확하게 바꿉니다.

`transfer_event_preseason=1`은 이제:

> **우리의 dated historical source에서 cutoff 이전이라고 확인된 관련 transfer event가 존재한다**

를 의미합니다.

날짜를 확인하지 못한 summer event는 0으로 '이적 없음'이라고 확정하는 것이 아니라
**모델 feature에는 사용하지 않고 audit에만 남깁니다.**

또한:

- transfer event가 있었지만 최종 팀이 그대로일 수 있음
- 여러 번 이동 후 원래 팀으로 돌아올 수도 있음

따라서 event와 최종 team change를 분리합니다.

In [25]:
snapshot[
    "transfer_event_preseason"
] = (
    snapshot[
        "confirmed_preseason_event_count"
    ]
    .fillna(0)
    .gt(0)
    .astype(int)
)

snapshot[
    "norm_destination_team"
] = (
    snapshot[
        "destination_team"
    ]
    .map(normalize_club)
)

# 최종 cutoff 시점 팀이 원래 팀과 다른가?
snapshot[
    "changed_team_preseason"
] = (
    snapshot.apply(
        lambda r: int(
            club_similarity(
                r["team"],
                r[
                    "destination_team"
                ],
            )
            < SAME_CLUB_SCORE
        ),
        axis=1,
    )
)

# league 표기 다시 canonicalize
snapshot[
    "destination_league"
] = (
    snapshot[
        "destination_league"
    ]
    .map(canonicalize_league)
)

# 이적이 없거나 최종 팀이 그대로면
# 현재 Big5 league를 유지
same_final_team = (
    snapshot[
        "changed_team_preseason"
    ].eq(0)
)

snapshot.loc[
    same_final_team,
    "destination_league",
] = snapshot.loc[
    same_final_team,
    "league",
]

snapshot.loc[
    same_final_team,
    "destination_country",
] = (
    snapshot.loc[
        same_final_team,
        "league",
    ]
    .map(
        LEAGUE_COUNTRY
    )
)

snapshot[
    "destination_in_big5"
] = (
    snapshot[
        "destination_league"
    ]
    .isin(
        BIG5_LEAGUES
    )
    .astype(int)
)

# 같은 리그 이적
snapshot[
    "same_league_transfer"
] = 0

changed_mask = (
    snapshot[
        "changed_team_preseason"
    ].eq(1)
)

snapshot.loc[
    changed_mask,
    "same_league_transfer",
] = (
    snapshot.loc[
        changed_mask,
        "destination_league",
    ]
    .eq(
        snapshot.loc[
            changed_mask,
            "league",
        ]
    )
    .astype(int)
)

snapshot[
    "league_changed"
] = (
    changed_mask
    & snapshot[
        "same_league_transfer"
    ].eq(0)
).astype(int)

current_country = (
    snapshot[
        "league"
    ]
    .map(
        LEAGUE_COUNTRY
    )
)

snapshot[
    "country_changed"
] = 0

country_known = (
    changed_mask
    & snapshot[
        "destination_country"
    ].notna()
)

snapshot.loc[
    country_known,
    "country_changed",
] = (
    snapshot.loc[
        country_known,
        "destination_country",
    ]
    .astype(str)
    .str.lower()
    .ne(
        current_country.loc[
            country_known
        ]
        .astype(str)
        .str.lower()
    )
    .astype(int)
)

snapshot[
    "is_loan_preseason"
] = 0

selected_event_mask = (
    snapshot[
        "selected_event_id"
    ].notna()
)

snapshot.loc[
    selected_event_mask,
    "is_loan_preseason",
] = (
    snapshot.loc[
        selected_event_mask,
        "tr_is_loan",
    ]
    .fillna(0)
    .astype(int)
)

snapshot[
    "transfer_date_preseason"
] = pd.NaT

snapshot.loc[
    selected_event_mask,
    "transfer_date_preseason",
] = (
    snapshot.loc[
        selected_event_mask,
        "tr_transfer_date",
    ]
)

snapshot[
    "days_since_transfer"
] = (
    snapshot[
        "cutoff_date"
    ]
    - snapshot[
        "transfer_date_preseason"
    ]
).dt.days

snapshot[
    "transfer_fee_eur"
] = np.nan

snapshot.loc[
    selected_event_mask,
    "transfer_fee_eur",
] = snapshot.loc[
    selected_event_mask,
    "tr_transfer_fee_eur",
]

snapshot[
    "transfer_market_value_eur"
] = np.nan

snapshot.loc[
    selected_event_mask,
    "transfer_market_value_eur",
] = snapshot.loc[
    selected_event_mask,
    "tr_transfer_market_value_eur",
]

snapshot[
    "transfer_fee_known"
] = (
    snapshot[
        "transfer_fee_eur"
    ].notna()
    & selected_event_mask
).astype(int)

snapshot[
    "transfer_fee_positive"
] = (
    snapshot[
        "transfer_fee_eur"
    ]
    .fillna(0)
    .gt(0)
    .astype(int)
)

snapshot[
    "log_transfer_fee"
] = np.log1p(
    snapshot[
        "transfer_fee_eur"
    ]
    .clip(lower=0)
)

snapshot[
    "fee_to_transfer_market_value_ratio"
] = np.where(
    snapshot[
        "transfer_market_value_eur"
    ] > 0,
    (
        snapshot[
            "transfer_fee_eur"
        ]
        / snapshot[
            "transfer_market_value_eur"
        ]
    ),
    np.nan,
)

display(
    snapshot.loc[
        snapshot[
            "transfer_event_preseason"
        ].eq(1),
        [
            "player",
            "season",
            "team",
            "destination_team",
            "transfer_date_preseason",
            "cutoff_date",
            "confirmed_preseason_event_count",
            "preseason_team_transition_count",
            "changed_team_preseason",
            "destination_league",
            "destination_in_big5",
            "is_loan_preseason",
            "transfer_fee_eur",
        ],
    ].head(25)
)

,player,season,team,destination_team,transfer_date_preseason,cutoff_date,confirmed_preseason_event_count,preseason_team_transition_count,changed_team_preseason,destination_league,destination_in_big5,is_loan_preseason,transfer_fee_eur
3105,James Milner,2003-2004,Leeds United,Newcastle United,2004-07-02,2004-07-31,1,1,1,Premier League,1,0,7400000.0
5152,Lukas Podolski,2005-2006,Köln,Bayern Munich,2006-07-10,2006-07-31,1,1,1,Bundesliga,1,0,10000000.0
5407,Santi Cazorla,2005-2006,Villarreal,Recreativo Huelva,2006-07-07,2006-07-31,1,1,1,La Liga,1,0,400000.0
5579,André-Pierre Gignac,2006-2007,Lorient,FC Toulouse,2007-07-01,2007-07-31,1,1,1,Ligue 1,1,0,4500000.0
6326,Raúl García,2006-2007,Osasuna,Atlético de Madrid,2007-07-01,2007-07-31,1,1,1,La Liga,1,0,13000000.0
6375,Santi Cazorla,2006-2007,Recreativo,Villarreal CF,2007-07-01,2007-07-31,1,1,1,La Liga,1,0,1200000.0
6430,Steven Davis,2006-2007,Aston Villa,Fulham FC,2007-07-01,2007-07-31,1,1,1,Premier League,1,0,5900000.0
7073,Kévin Gameiro,2007-2008,Strasbourg,FC Lorient,2008-07-01,2008-07-31,1,1,1,Ligue 1,1,0,3000000.0
7231,Miralem Pjanić,2007-2008,Metz,Olympique Lyon,2008-07-01,2008-07-31,1,1,1,Ligue 1,1,0,7500000.0
7554,Álvaro Negredo,2008-2009,Almería,Real Madrid,2009-07-01,2009-07-31,1,1,1,La Liga,1,0,5000000.0


# Part E. Historical Market Value — Point-in-time Join

## 21. player_valuations 정리

시장가치에서 가장 중요한 원칙:

```text
valuation_date <= cutoff_date
```

현재 최신 시장가치를 과거 모든 시즌에 붙이면 심각한 미래 누수입니다.

아래에서는 각 선수-시즌 row마다:

- cutoff 직전 최신 시장가치
- cutoff - 6개월 시점에서 알 수 있었던 최신 시장가치
- cutoff - 12개월 시점에서 알 수 있었던 최신 시장가치
- cutoff까지의 역사적 peak

를 계산합니다.

In [26]:
valuations = pd.DataFrame({
    "tm_player_id": pd.to_numeric(
        valuations_raw[
            VALUATION_PLAYER_ID_COL
        ],
        errors="coerce",
    ),
    "valuation_date": (
        pd.to_datetime(
            valuations_raw[
                VALUATION_DATE_COL
            ],
            errors="coerce",
        )
        .astype("datetime64[ns]")
    ),
    "market_value_eur": pd.to_numeric(
        valuations_raw[
            VALUATION_VALUE_COL
        ],
        errors="coerce",
    ),
})

valuations = (
    valuations
    .dropna(
        subset=[
            "tm_player_id",
            "valuation_date",
            "market_value_eur",
        ]
    )
    .copy()
)

valuations[
    "tm_player_id"
] = (
    valuations[
        "tm_player_id"
    ]
    .astype(int)
)

valuations = (
    valuations
    .sort_values(
        [
            "tm_player_id",
            "valuation_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "tm_player_id",
            "valuation_date",
        ],
        keep="last",
    )
)

valuations[
    "market_value_peak_to_date_eur"
] = (
    valuations
    .groupby(
        "tm_player_id"
    )
    ["market_value_eur"]
    .cummax()
)

print(
    "Valuation rows:",
    len(valuations),
)

print(
    "Date range:",
    valuations["valuation_date"].min(),
    "~",
    valuations["valuation_date"].max(),
)

Valuation rows: 656301
Date range: 2000-01-20 00:00:00 ~ 2026-06-12 00:00:00


## 22. Generic point-in-time lookup 함수

In [27]:
def point_in_time_valuation_lookup(
    base_df,
    query_date_series,
    prefix,
):
    left = base_df[
        [
            "row_id",
            "tm_player_id",
        ]
    ].copy()

    # pandas merge_asof는 양쪽 datetime dtype의 해상도까지 같아야 합니다.
    # DuckDB/Pandas 버전에 따라 datetime64[us] / datetime64[ns]가 섞일 수 있어
    # 명시적으로 모두 ns로 통일합니다.
    left["query_date"] = (
        pd.to_datetime(
            query_date_series,
            errors="coerce",
        )
        .astype("datetime64[ns]")
    )

    valid_left = (
        left[
            "tm_player_id"
        ].notna()
        & left[
            "query_date"
        ].notna()
    )

    query = (
        left.loc[
            valid_left
        ]
        .copy()
    )

    if query.empty:
        return pd.DataFrame({
            "row_id": base_df["row_id"],
            f"{prefix}_value_eur": np.nan,
            f"{prefix}_valuation_date": pd.NaT,
            f"{prefix}_peak_eur": np.nan,
        })

    query[
        "tm_player_id"
    ] = (
        query[
            "tm_player_id"
        ]
        .astype(int)
    )

    right = valuations[
        [
            "tm_player_id",
            "valuation_date",
            "market_value_eur",
            "market_value_peak_to_date_eur",
        ]
    ].copy()

    # merge_asof dtype 일치 보장
    right["valuation_date"] = (
        pd.to_datetime(
            right["valuation_date"],
            errors="coerce",
        )
        .astype("datetime64[ns]")
    )

    # 혹시 입력 dataframe에 object/nullable 변형이 섞여 있어도 정규화
    query["query_date"] = (
        pd.to_datetime(
            query["query_date"],
            errors="coerce",
        )
        .astype("datetime64[ns]")
    )

    # merge_asof는 on key가 정렬되어 있어야 함
    query = query.sort_values(
        [
            "query_date",
            "tm_player_id",
        ]
    )

    right = right.sort_values(
        [
            "valuation_date",
            "tm_player_id",
        ]
    )

    merged = pd.merge_asof(
        query,
        right,
        left_on="query_date",
        right_on="valuation_date",
        by="tm_player_id",
        direction="backward",
        allow_exact_matches=True,
    )

    result = (
        merged[
            [
                "row_id",
                "market_value_eur",
                "valuation_date",
                "market_value_peak_to_date_eur",
            ]
        ]
        .rename(
            columns={
                "market_value_eur": f"{prefix}_value_eur",
                "valuation_date": f"{prefix}_valuation_date",
                "market_value_peak_to_date_eur": f"{prefix}_peak_eur",
            }
        )
    )

    all_rows = pd.DataFrame({
        "row_id": base_df[
            "row_id"
        ]
    })

    result = all_rows.merge(
        result,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    return result

## 23. cutoff / 6개월 전 / 12개월 전 시장가치 붙이기

In [28]:
# merge_asof 직전 datetime 해상도 확인
print("cutoff_date dtype:", snapshot["cutoff_date"].dtype)
print("valuation_date dtype:", valuations["valuation_date"].dtype)

assert str(
    pd.to_datetime(
        snapshot["cutoff_date"],
        errors="coerce",
    ).astype("datetime64[ns]").dtype
) == "datetime64[ns]"

assert str(
    pd.to_datetime(
        valuations["valuation_date"],
        errors="coerce",
    ).astype("datetime64[ns]").dtype
) == "datetime64[ns]"

print("✅ datetime resolution normalized to ns")


cutoff_date dtype: datetime64[ns]
valuation_date dtype: datetime64[ns]
✅ datetime resolution normalized to ns


In [29]:
mv_now = point_in_time_valuation_lookup(
    snapshot,
    snapshot["cutoff_date"],
    prefix="mv_preseason",
)

mv_6m = point_in_time_valuation_lookup(
    snapshot,
    (
        snapshot["cutoff_date"]
        - pd.Timedelta(days=183)
    ),
    prefix="mv_6m",
)

mv_12m = point_in_time_valuation_lookup(
    snapshot,
    (
        snapshot["cutoff_date"]
        - pd.Timedelta(days=365)
    ),
    prefix="mv_12m",
)

for table in [
    mv_now,
    mv_6m,
    mv_12m,
]:
    snapshot = snapshot.merge(
        table,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

print(
    "Raw preseason valuation coverage:",
    f"{snapshot['mv_preseason_value_eur'].notna().mean():.2%}"
)

Raw preseason valuation coverage: 69.13%


## 24. Market Value Feature 생성

cutoff 직전 valuation이 없지만
cutoff 이전에 실제 이적이 있었고 그 event에 market value가 있다면
그 값을 fallback으로 사용할 수 있습니다.

다만 provenance를 남깁니다.

```text
market_value_source
= historical_valuation
  / transfer_event_fallback
  / missing
```

In [30]:
snapshot[
    "market_value_preseason_eur"
] = snapshot[
    "mv_preseason_value_eur"
]

snapshot[
    "market_value_source"
] = np.where(
    snapshot[
        "mv_preseason_value_eur"
    ].notna(),
    "historical_valuation",
    "missing",
)

fallback_mv_mask = (
    snapshot[
        "market_value_preseason_eur"
    ].isna()
    & snapshot[
        "transfer_event_preseason"
    ].eq(1)
    & snapshot[
        "transfer_market_value_eur"
    ].notna()
)

snapshot.loc[
    fallback_mv_mask,
    "market_value_preseason_eur",
] = snapshot.loc[
    fallback_mv_mask,
    "transfer_market_value_eur",
]

snapshot.loc[
    fallback_mv_mask,
    "market_value_source",
] = "transfer_event_fallback"

snapshot[
    "market_value_known"
] = (
    snapshot[
        "market_value_preseason_eur"
    ].notna()
).astype(int)

snapshot[
    "log_market_value"
] = np.log1p(
    snapshot[
        "market_value_preseason_eur"
    ].clip(lower=0)
)

snapshot[
    "market_value_6m_eur"
] = snapshot[
    "mv_6m_value_eur"
]

snapshot[
    "market_value_12m_eur"
] = snapshot[
    "mv_12m_value_eur"
]

snapshot[
    "market_value_change_6m_eur"
] = (
    snapshot[
        "market_value_preseason_eur"
    ]
    - snapshot[
        "market_value_6m_eur"
    ]
)

snapshot[
    "market_value_change_12m_eur"
] = (
    snapshot[
        "market_value_preseason_eur"
    ]
    - snapshot[
        "market_value_12m_eur"
    ]
)

snapshot[
    "market_value_growth_6m"
] = np.where(
    snapshot[
        "market_value_6m_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_6m_eur"
        ]
        - 1
    ),
    np.nan,
)

snapshot[
    "market_value_growth_12m"
] = np.where(
    snapshot[
        "market_value_12m_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_12m_eur"
        ]
        - 1
    ),
    np.nan,
)

snapshot[
    "market_value_peak_to_cutoff_eur"
] = snapshot[
    "mv_preseason_peak_eur"
]

snapshot[
    "market_value_vs_peak"
] = np.where(
    snapshot[
        "market_value_peak_to_cutoff_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_peak_to_cutoff_eur"
        ]
    ),
    np.nan,
)

snapshot[
    "valuation_age_days"
] = (
    snapshot[
        "cutoff_date"
    ]
    - snapshot[
        "mv_preseason_valuation_date"
    ]
).dt.days

# 같은 target season 내 상대적 market value
snapshot[
    "market_value_percentile"
] = (
    snapshot
    .groupby(
        "target_season"
    )
    ["market_value_preseason_eur"]
    .rank(
        pct=True,
        method="average",
    )
)

position_median = (
    snapshot
    .groupby(
        [
            "target_season",
            "position_group",
        ]
    )
    ["market_value_preseason_eur"]
    .transform("median")
)

snapshot[
    "market_value_vs_position_median"
] = np.where(
    position_median > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / position_median
    ),
    np.nan,
)

market_value_snapshots = (
    snapshot[
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "tm_player_id",
            "cutoff_date",
            "market_value_preseason_eur",
            "market_value_source",
            "mv_preseason_valuation_date",
            "market_value_6m_eur",
            "mv_6m_valuation_date",
            "market_value_12m_eur",
            "mv_12m_valuation_date",
            "market_value_peak_to_cutoff_eur",
            "market_value_growth_6m",
            "market_value_growth_12m",
            "market_value_vs_peak",
        ]
    ]
    .copy()
)

display(
    snapshot[
        [
            "target_season",
            "market_value_known",
        ]
    ]
    .groupby("target_season")
    .agg(
        n=("market_value_known", "size"),
        coverage=("market_value_known", "mean"),
    )
    .tail(15)
)

,n,coverage
target_season,,
2011-2012,1009,0.802775
2012-2013,1042,0.888676
2013-2014,1063,0.948260
2014-2015,1065,0.953991
2015-2016,1093,0.949680
2016-2017,991,0.954591
2017-2018,993,0.952669
2018-2019,954,0.949686
2019-2020,956,0.947699


# Part F. New Team Environment

## 25. 현재 팀 직전 시즌 성적 연결

현재 선수 row가 2023-24라면
2023-24 팀 최종 성적은 시즌 종료 후 이미 알고 있는 정보입니다.

기존 Exp9Cb에서도 사용했던:

- team_rank_pct
- team_points_per_game
- team_goal_diff_per_game

를 old team 기준으로 다시 붙입니다.

In [31]:
alias_map = dict(
    zip(
        team_alias[
            "player_data_team"
        ],
        team_alias[
            "standings_team"
        ],
    )
)

alias_map.update({
    "Gladbach": "M'gladbach",
    "Luton Town": "Luton",
})

snapshot[
    "old_team_stats_name"
] = (
    snapshot["team"]
    .replace(alias_map)
)

TEAM_STRENGTH_COLS = [
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game",
]

team_strength = (
    team_stats[
        [
            "league",
            "season",
            "team_name",
        ]
        + TEAM_STRENGTH_COLS
    ]
    .drop_duplicates(
        subset=[
            "league",
            "season",
            "team_name",
        ]
    )
    .copy()
)

team_strength[
    "league"
] = (
    team_strength[
        "league"
    ]
    .map(
        canonicalize_league
    )
)

old_strength = (
    team_strength
    .rename(
        columns={
            "team_name": "old_team_stats_name",
            "team_rank_pct": "old_team_rank_pct",
            "team_points_per_game": "old_team_points_per_game",
            "team_goal_diff_per_game": "old_team_goal_diff_per_game",
        }
    )
)

snapshot = snapshot.merge(
    old_strength,
    on=[
        "league",
        "season",
        "old_team_stats_name",
    ],
    how="left",
    validate="many_to_one",
)

old_team_match_rate = (
    snapshot[
        [
            "old_team_rank_pct",
            "old_team_points_per_game",
            "old_team_goal_diff_per_game",
        ]
    ]
    .notna()
    .all(axis=1)
    .mean()
)

print(
    "Old team strength match:",
    f"{old_team_match_rate:.2%}"
)

assert (
    old_team_match_rate
    >= MIN_OLD_TEAM_MATCH_RATE
), (
    "기존 팀 strength 매칭률이 낮습니다. "
    "team alias를 확인하세요."
)

Old team strength match: 100.00%


## 26. v2 — 새 팀의 직전 시즌 T 성적 연결

v1의 coverage가 낮았던 원인 중 하나는:

- `Laliga` vs `La Liga`
- `Borussia Dortmund` vs `Dortmund`
- `Atalanta BC` vs `Atalanta`
- `Manchester United` vs `Man United`

같은 표기 문제였습니다.

v2에서는:

1. league canonicalization
2. 기존 `team_name_alias_map`
3. generic club token 제거
4. club similarity
5. 같은 league 내 best/second-best margin

을 함께 사용합니다.

여전히 직전 시즌 Big5에 없었던 승격팀/외부리그 팀은
정상적인 missing으로 남깁니다.

In [32]:
team_strength_lookup = (
    team_strength
    .copy()
)

team_strength_lookup[
    "norm_team_name"
] = (
    team_strength_lookup[
        "team_name"
    ]
    .map(
        normalize_club
    )
)

# 기존 player_data_team ↔ standings_team alias도
# destination matching 후보로 사용
standings_alias_records = []

for row in team_alias.itertuples(
    index=False
):
    standings_alias_records.append({
        "player_data_team": row.player_data_team,
        "standings_team": row.standings_team,
        "norm_player_team": normalize_club(
            row.player_data_team
        ),
        "norm_standings_team": normalize_club(
            row.standings_team
        ),
    })

standings_alias_df = pd.DataFrame(
    standings_alias_records
)

team_strength_groups = {
    key: group.reset_index(
        drop=True
    )
    for key, group
    in team_strength_lookup.groupby(
        [
            "season",
            "league",
        ]
    )
}


def destination_candidate_score(
    destination_team,
    candidate_team,
):
    score = club_similarity(
        destination_team,
        candidate_team,
    )

    # transfermarkt 이름이 기존 player_data_team alias와
    # 더 잘 맞는 경우 해당 standings 이름 점수도 활용
    alias_rows = (
        standings_alias_df.loc[
            standings_alias_df[
                "standings_team"
            ].eq(
                candidate_team
            )
        ]
    )

    for alias_row in alias_rows.itertuples():
        score = max(
            score,
            club_similarity(
                destination_team,
                alias_row.player_data_team,
            ),
        )

    return float(score)


def match_destination_team_stats(
    season,
    destination_team,
    destination_league,
):
    if (
        pd.isna(destination_team)
        or pd.isna(destination_league)
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "not_big5_or_unknown",
        }

    destination_league = (
        canonicalize_league(
            destination_league
        )
    )

    if (
        destination_league
        not in BIG5_LEAGUES
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "not_big5_or_unknown",
        }

    destination_team = (
        MANUAL_DESTINATION_TEAM_OVERRIDES
        .get(
            destination_team,
            destination_team,
        )
    )

    key = (
        str(season),
        str(destination_league),
    )

    candidates = (
        team_strength_groups.get(
            key
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "no_team_stats_candidates",
        }

    scored = candidates[
        [
            "team_name",
        ]
    ].copy()

    scored[
        "score"
    ] = scored[
        "team_name"
    ].map(
        lambda candidate:
            destination_candidate_score(
                destination_team,
                candidate,
            )
    )

    scored = scored.sort_values(
        "score",
        ascending=False,
    ).reset_index(
        drop=True
    )

    best = scored.iloc[0]

    second_score = (
        float(
            scored.iloc[1][
                "score"
            ]
        )
        if len(scored) > 1
        else 0.0
    )

    best_score = float(
        best["score"]
    )

    margin = (
        best_score
        - second_score
    )

    if (
        best_score
        >= NEW_TEAM_MIN_SCORE
        and (
            best_score >= 0.88
            or margin
            >= NEW_TEAM_MIN_MARGIN
        )
    ):
        method = (
            "exact_or_alias"
            if best_score >= 0.98
            else "scored_team_match"
        )

        return {
            "new_team_stats_name": best[
                "team_name"
            ],
            "new_team_match_score": best_score,
            "new_team_second_score": second_score,
            "new_team_match_margin": margin,
            "new_team_match_method": method,
        }

    return {
        "new_team_stats_name": np.nan,
        "new_team_match_score": best_score,
        "new_team_second_score": second_score,
        "new_team_match_margin": margin,
        "new_team_match_method": "unresolved_team_name",
    }


new_team_match_rows = []
cache = {}

for row in snapshot[
    [
        "row_id",
        "season",
        "team",
        "destination_team",
        "destination_league",
        "changed_team_preseason",
        "old_team_stats_name",
    ]
].itertuples(
    index=False
):
    if (
        row.changed_team_preseason
        == 0
    ):
        result = {
            "new_team_stats_name": row.old_team_stats_name,
            "new_team_match_score": 1.0,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "same_team_copy_old",
        }

    else:
        cache_key = (
            row.season,
            row.destination_team,
            canonicalize_league(
                row.destination_league
            ),
        )

        if (
            cache_key
            not in cache
        ):
            cache[
                cache_key
            ] = (
                match_destination_team_stats(
                    season=row.season,
                    destination_team=row.destination_team,
                    destination_league=row.destination_league,
                )
            )

        result = cache[
            cache_key
        ]

    new_team_match_rows.append({
        "row_id": row.row_id,
        **result,
    })


new_team_matches = (
    pd.DataFrame(
        new_team_match_rows
    )
)

snapshot = snapshot.merge(
    new_team_matches,
    on="row_id",
    how="left",
    validate="one_to_one",
)

new_strength = (
    team_strength
    .rename(
        columns={
            "league": "new_team_strength_league",
            "team_name": "new_team_stats_name",
            "team_rank_pct": "new_team_prev_rank_pct",
            "team_points_per_game": "new_team_prev_points_per_game",
            "team_goal_diff_per_game": "new_team_prev_goal_diff_per_game",
        }
    )
)

snapshot[
    "new_team_strength_league"
] = (
    snapshot[
        "destination_league"
    ]
    .map(
        canonicalize_league
    )
)

snapshot = snapshot.merge(
    new_strength,
    on=[
        "new_team_strength_league",
        "season",
        "new_team_stats_name",
    ],
    how="left",
    validate="many_to_one",
)

snapshot[
    "new_team_strength_missing"
] = (
    snapshot[
        [
            "new_team_prev_rank_pct",
            "new_team_prev_points_per_game",
            "new_team_prev_goal_diff_per_game",
        ]
    ]
    .isna()
    .any(axis=1)
    .astype(int)
)

snapshot[
    "team_rank_change"
] = (
    snapshot[
        "new_team_prev_rank_pct"
    ]
    - snapshot[
        "old_team_rank_pct"
    ]
)

snapshot[
    "team_points_change"
] = (
    snapshot[
        "new_team_prev_points_per_game"
    ]
    - snapshot[
        "old_team_points_per_game"
    ]
)

snapshot[
    "team_goal_diff_change"
] = (
    snapshot[
        "new_team_prev_goal_diff_per_game"
    ]
    - snapshot[
        "old_team_goal_diff_per_game"
    ]
)

print(
    "Changed-team rows:",
    int(
        snapshot[
            "changed_team_preseason"
        ].sum()
    ),
)

changed_big5 = (
    snapshot[
        "changed_team_preseason"
    ].eq(1)
    & snapshot[
        "destination_in_big5"
    ].eq(1)
)

print(
    "Changed team → Big5 rows:",
    int(
        changed_big5.sum()
    ),
)

if changed_big5.any():
    coverage = (
        snapshot.loc[
            changed_big5,
            "new_team_strength_missing",
        ]
        .eq(0)
        .mean()
    )

    print(
        "New-team previous-season strength coverage:",
        f"{coverage:.2%}"
    )

    display(
        snapshot.loc[
            changed_big5,
            [
                "destination_team",
                "destination_league",
                "new_team_stats_name",
                "new_team_match_score",
                "new_team_match_margin",
                "new_team_match_method",
                "new_team_strength_missing",
            ],
        ]
        .head(30)
    )

Changed-team rows: 1221
Changed team → Big5 rows: 780
New-team previous-season strength coverage: 88.46%


,destination_team,destination_league,new_team_stats_name,new_team_match_score,new_team_match_margin,new_team_match_method,new_team_strength_missing
3105,Newcastle United,Premier League,Newcastle,0.920000,0.192727,scored_team_match,0
5152,Bayern Munich,Bundesliga,Bayern Munich,1.000000,0.500000,exact_or_alias,0
5407,Recreativo Huelva,La Liga,NaN,0.400000,0.015385,unresolved_team_name,1
5579,FC Toulouse,Ligue 1,Toulouse,0.980000,0.627059,exact_or_alias,0
6326,Atlético de Madrid,La Liga,Ath Madrid,0.980000,0.359310,exact_or_alias,0
6375,Villarreal CF,La Liga,Villarreal,0.980000,0.480000,exact_or_alias,0
6430,Fulham FC,Premier League,Fulham,0.980000,0.627059,exact_or_alias,0
7073,FC Lorient,Ligue 1,Lorient,0.980000,0.551429,exact_or_alias,0
7231,Olympique Lyon,Ligue 1,NaN,0.444444,0.063492,unresolved_team_name,1
7554,Real Madrid,La Liga,Real Madrid,1.000000,0.238095,exact_or_alias,0


# Part G. Leakage Audit

## 27. Leakage assertions

이 셀은 단순한 확인용 출력이 아니라
**실제로 잘못된 row가 있으면 Notebook을 중단**합니다.

In [33]:
# 1) model에 붙인 selected transfer 날짜는 반드시 cutoff 이하
bad_transfer = snapshot.loc[
    snapshot[
        "transfer_event_preseason"
    ].eq(1)
    & (
        snapshot[
            "transfer_date_preseason"
        ]
        > snapshot[
            "cutoff_date"
        ]
    )
]

assert bad_transfer.empty, (
    "LEAKAGE: cutoff 이후 transfer가 "
    "preseason feature에 포함되었습니다."
)

# 2) transfer_event_preseason=1이면 날짜가 반드시 존재
bad_missing_transfer_date = (
    snapshot.loc[
        snapshot[
            "transfer_event_preseason"
        ].eq(1)
        & snapshot[
            "transfer_date_preseason"
        ].isna()
    ]
)

assert (
    bad_missing_transfer_date.empty
), (
    "LEAKAGE/QUALITY: 날짜 없는 transfer가 "
    "preseason feature에 포함되었습니다."
)

# 3) destination league는 canonical label만 사용
known_destination_league = (
    snapshot[
        "destination_league"
    ]
    .dropna()
)

bad_big5_spellings = (
    known_destination_league[
        known_destination_league
        .astype(str)
        .str.lower()
        .eq("laliga")
    ]
)

assert (
    bad_big5_spellings.empty
), (
    "QUALITY: Laliga 표기가 canonicalize되지 않았습니다."
)

# 4) historical valuation은 cutoff 이하
bad_valuation = snapshot.loc[
    snapshot[
        "mv_preseason_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_preseason_valuation_date"
        ]
        > snapshot[
            "cutoff_date"
        ]
    )
]

assert bad_valuation.empty, (
    "LEAKAGE: cutoff 이후 시장가치가 포함되었습니다."
)

# 5) 6m valuation
bad_6m = snapshot.loc[
    snapshot[
        "mv_6m_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_6m_valuation_date"
        ]
        > (
            snapshot[
                "cutoff_date"
            ]
            - pd.Timedelta(
                days=183
            )
        )
    )
]

assert bad_6m.empty, (
    "LEAKAGE: 6m snapshot 날짜 규칙 위반"
)

# 6) 12m valuation
bad_12m = snapshot.loc[
    snapshot[
        "mv_12m_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_12m_valuation_date"
        ]
        > (
            snapshot[
                "cutoff_date"
            ]
            - pd.Timedelta(
                days=365
            )
        )
    )
]

assert bad_12m.empty, (
    "LEAKAGE: 12m snapshot 날짜 규칙 위반"
)

# 7) games 기반 cutoff는 첫 Big5 경기 이전
games_cutoff_rows = cutoff_table[
    cutoff_table[
        "cutoff_source"
    ].eq(
        "duckdb_big5_first_game"
    )
]

assert (
    games_cutoff_rows[
        "cutoff_date"
    ]
    < games_cutoff_rows[
        "first_big5_game_date"
    ]
).all()

# 8) Final Test row는 feature-build snapshot에 존재해야 하지만
#    정답(label)은 아직 숨겨져 있어야 함
final_test_feature_rows = snapshot.loc[
    snapshot[
        "season"
    ]
    .astype(str)
    .eq("2024-2025")
]

assert not final_test_feature_rows.empty, (
    "Final Test input season 2024-2025가 "
    "feature-build snapshot에 없습니다."
)

for label_col in [
    "matched_next",
    "next_goals",
    "next_10plus",
]:
    if label_col in final_test_feature_rows.columns:
        assert (
            final_test_feature_rows[
                label_col
            ]
            .isna()
            .all()
        ), (
            f"LEAKAGE: Final Test {label_col}가 "
            "feature 생성 시점에 노출되었습니다."
        )

print(
    "✅ v2 Leakage / canonicalization assertions passed."
)

✅ v2 Leakage / canonicalization assertions passed.


## 28. Feature construction과 Label 분리 확인

`matched_next`, `next_goals`, `next_10plus`는 최종 데이터에 보존하지만
외부 feature 생성 로직에는 사용하지 않았습니다.

다음 Notebook에서 목적에 따라:

```text
Model A:
전체 row + matched_next

Model B:
matched_next=True + next_goals
```

로 나눌 수 있습니다.

In [34]:
LABEL_COLS = [
    col
    for col in [
        "matched_next",
        "next_goals",
        "next_10plus",
    ]
    if col in snapshot.columns
]

print("Labels preserved:", LABEL_COLS)

Labels preserved: ['matched_next', 'next_goals', 'next_10plus']


# Part I. Development audited label 복원

여기까지 external feature integration은 Test label을 보지 않고 끝났습니다.

이제:

- Train + Validation에는 09-01A audited label 적용
- Final Test label은 아직 숨김

상태로 최종 모델을 학습합니다.

In [35]:
audited_dev = pd.read_csv(
    AUDITED_DEV_PATH,
    low_memory=False,
)

required_audited = {
    "row_id",
    "player",
    "season",
    "team",
    "matched_next_audited",
    "next_goals_audited",
}

assert required_audited.issubset(
    audited_dev.columns
)


# row_id 정렬이 기존 dev snapshot과 같은지 identity check
feature_dev_identity = (
    snapshot.loc[
        snapshot[
            "row_id"
        ]
        < DEV_RAW_N,
        [
            "row_id",
            "player",
            "season",
            "team",
        ],
    ]
    .sort_values(
        "row_id"
    )
    .reset_index(
        drop=True
    )
)

audit_identity = (
    audited_dev[
        [
            "row_id",
            "player",
            "season",
            "team",
        ]
    ]
    .sort_values(
        "row_id"
    )
    .reset_index(
        drop=True
    )
)

assert (
    len(
        feature_dev_identity
    )
    == len(
        audit_identity
    )
), (
    "Development row 수가 09-01A snapshot과 다릅니다."
)

identity_ok = (
    feature_dev_identity[
        [
            "player",
            "season",
            "team",
        ]
    ]
    .astype(str)
    .eq(
        audit_identity[
            [
                "player",
                "season",
                "team",
            ]
        ]
        .astype(str)
    )
    .all()
    .all()
)

assert identity_ok, (
    "row_id identity가 09-01A와 일치하지 않습니다."
)


audit_labels = (
    audited_dev[
        [
            "row_id",
            "matched_next_audited",
            "next_goals_audited",
            "next_10plus_audited",
        ]
    ]
    .copy()
)


snapshot = snapshot.merge(
    audit_labels,
    on="row_id",
    how="left",
    validate="one_to_one",
)


development_mask = (
    snapshot[
        "row_id"
    ]
    < DEV_RAW_N
)

final_test_mask = (
    snapshot[
        "season"
    ]
    .astype(str)
    .eq(
        "2024-2025"
    )
)


assert snapshot.loc[
    development_mask,
    "matched_next_audited",
].notna().all()

assert snapshot.loc[
    final_test_mask,
    "next_goals",
].isna().all()


print(
    "Development audited rows:",
    development_mask.sum(),
)

print(
    "Final Test feature rows:",
    final_test_mask.sum(),
)

print(
    "✅ Final Test labels still hidden."
)

Development audited rows: 23353
Final Test feature rows: 926
✅ Final Test labels still hidden.


# Part J. Historical Features

09/10과 같은 방식으로 과거 3시즌 득점 history를 생성합니다.

Test 2024-25 row의 history는 2023-24 이전 정보만 사용합니다.

In [36]:
snapshot = (
    snapshot
    .sort_values(
        [
            "player",
            "season_start",
        ]
    )
    .reset_index(
        drop=True
    )
)

# 중요:
# 위 sort/reset_index 이후에는 이전에 만들어 둔 boolean mask의
# 행 위치가 더 이상 snapshot과 일치하지 않을 수 있습니다.
# row_id / season 기준으로 mask를 반드시 다시 계산합니다.
development_mask = (
    snapshot[
        "row_id"
    ]
    < DEV_RAW_N
)

final_test_mask = (
    snapshot[
        "season"
    ]
    .astype(str)
    .eq(
        "2024-2025"
    )
)

assert not (
    development_mask
    & final_test_mask
).any(), (
    "Development/Test mask가 겹칩니다."
)

print(
    "Masks recomputed after sort:",
    "development=",
    int(
        development_mask.sum()
    ),
    "test=",
    int(
        final_test_mask.sum()
    ),
)

g = snapshot.groupby(
    "player",
    sort=False,
)

snapshot[
    "goals_3yr_mean"
] = g[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

snapshot[
    "goals_per90_3yr_mean"
] = g[
    "goals_per90"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

snapshot[
    "goals_3yr_max"
] = g[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .max()
)


HISTORICAL = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

snapshot[
    HISTORICAL
] = (
    snapshot[
        HISTORICAL
    ]
    .fillna(0.0)
)

Masks recomputed after sort: development= 23353 test= 926


# Part K. Frozen FULL Features

In [37]:
BASE = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

TEAM = [
    "old_team_rank_pct",
    "old_team_points_per_game",
    "old_team_goal_diff_per_game",
]

MARKET = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

TRANSFER = [
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "is_loan_preseason",
    "days_since_transfer",
]

NEW_TEAM = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

MODEL_A_CONTEXT = [
    "transfer_event_preseason",
    "destination_in_big5",
]

CATEGORICAL = [
    "league",
    "position_group",
]

S3_NUMERIC = (
    BASE
    + HISTORICAL
    + TEAM
    + MARKET
    + TRANSFER
    + NEW_TEAM
)

MODEL_A_NUMERIC = (
    S3_NUMERIC
    + MODEL_A_CONTEXT
)

DIRECT_NUMERIC = (
    MODEL_A_NUMERIC
)


FORBIDDEN = {
    "matched_next",
    "matched_next_audited",
    "next_goals",
    "next_goals_audited",
    "next_10plus",
    "next_10plus_audited",
}

assert not (
    FORBIDDEN
    & set(
        MODEL_A_NUMERIC
    )
)

assert not (
    FORBIDDEN
    & set(
        S3_NUMERIC
    )
)

print(
    "✅ Frozen FULL features loaded."
)

✅ Frozen FULL features loaded.


# Part L. Train / Test 분리

여기서도 Test label은 아직 붙이지 않습니다.

In [38]:
# stale boolean-mask 위험을 피하기 위해
# 현재 snapshot에서 row_id / season 조건을 직접 다시 사용합니다.
train_dev = (
    snapshot[
        snapshot[
            "row_id"
        ].lt(
            DEV_RAW_N
        )
        & snapshot[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

final_test_features = (
    snapshot[
        snapshot[
            "season"
        ]
        .astype(str)
        .eq(
            "2024-2025"
        )
    ]
    .copy()
)


# Development label은 전부 채워져 있어야 합니다.
missing_audited = (
    train_dev[
        "matched_next_audited"
    ]
    .isna()
)

if missing_audited.any():
    display(
        train_dev.loc[
            missing_audited,
            [
                "row_id",
                "player",
                "team",
                "season",
                "target_season",
                "matched_next_audited",
                "next_goals_audited",
            ],
        ].head(20)
    )

    raise ValueError(
        "Development subset에 audited label NaN이 남아 있습니다. "
        "row_id / season split을 확인하세요."
    )


train_dev[
    "target_presence"
] = (
    train_dev[
        "matched_next_audited"
    ]
    .astype(int)
)

train_dev[
    "target_goals"
] = (
    train_dev[
        "next_goals_audited"
    ]
    .astype(float)
)


assert (
    final_test_features[
        "next_goals"
    ]
    .isna()
    .all()
)


print(
    "Final development rows:",
    len(
        train_dev
    ),
)

print(
    "Final test rows:",
    len(
        final_test_features
    ),
)

print(
    "Train last input season:",
    train_dev[
        "season"
    ].max(),
)

print(
    "Test input season:",
    final_test_features[
        "season"
    ].unique(),
)

Final development rows: 7572
Final test rows: 926
Train last input season: 2023-2024
Test input season: ['2024-2025']


# Part M. Model helpers

In [39]:
try:
    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )

except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "catboost",
        ]
    )

    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )


try:
    import torch

    USE_GPU = bool(
        torch.cuda.is_available()
    )

except ImportError:
    USE_GPU = False


DEVICE_PARAMS = (
    {
        "task_type": "GPU",
        "devices": "0",
    }
    if USE_GPU
    else {
        "task_type": "CPU",
    }
)


from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
    roc_auc_score,
)


def prepare_X(
    data,
    numeric_features,
):
    X = (
        data[
            numeric_features
            + CATEGORICAL
        ]
        .copy()
    )

    for col in (
        CATEGORICAL
    ):
        X[col] = (
            X[col]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
        )

    for col in (
        numeric_features
    ):
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce",
        )

    cat_idx = [
        X.columns.get_loc(
            col
        )
        for col in (
            CATEGORICAL
        )
    ]

    return (
        X,
        cat_idx,
    )


def split_final_calibration(
    data,
):
    last_season = (
        data[
            "season_start"
        ].max()
    )

    cal_train = (
        data[
            data[
                "season_start"
            ]
            < last_season
        ]
        .copy()
    )

    cal_val = (
        data[
            data[
                "season_start"
            ]
            == last_season
        ]
        .copy()
    )

    assert not (
        cal_train.empty
        or cal_val.empty
    )

    return (
        cal_train,
        cal_val,
    )

In [40]:
THRESHOLD_GRID = np.round(
    np.arange(
        0.20,
        0.951,
        0.01,
    ),
    2,
)


def classifier_threshold_metrics(
    y,
    prob,
    threshold,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    pred = (
        np.asarray(
            prob
        )
        >= threshold
    ).astype(int)

    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        y,
        pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    return {
        "macro_f1": (
            f1_score(
                y,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y,
                pred,
            )
        ),
        "exit_recall": (
            recall[0]
        ),
        "presence_recall": (
            recall[1]
        ),
    }


def choose_threshold(
    y,
    prob,
):
    rows = []

    for th in (
        THRESHOLD_GRID
    ):
        m = (
            classifier_threshold_metrics(
                y,
                prob,
                th,
            )
        )

        rows.append({
            "threshold": th,
            **m,
        })

    table = pd.DataFrame(
        rows
    )

    best = (
        table
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "presence_recall",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    return (
        float(
            best[
                "threshold"
            ]
        ),
        table,
    )

# Part N. Frozen Model A 최종 학습

Threshold와 iterations는 development 마지막 시즌(2023-24)을 calibration으로 사용해 정합니다.

**Final Test label은 아직 보지 않습니다.**

In [41]:
A_MAX_ITER = 2000
A_PATIENCE = 60

A_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
}


(
    a_cal_train,
    a_cal_val,
) = split_final_calibration(
    train_dev
)

(
    X_a_ct,
    a_cat_idx,
) = prepare_X(
    a_cal_train,
    MODEL_A_NUMERIC,
)

(
    X_a_cv,
    _,
) = prepare_X(
    a_cal_val,
    MODEL_A_NUMERIC,
)


a_selector = (
    CatBoostClassifier(
        iterations=A_MAX_ITER,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **A_PARAMS,
    )
)

a_selector.fit(
    X_a_ct,
    a_cal_train[
        "target_presence"
    ],
    cat_features=(
        a_cat_idx
    ),
    eval_set=(
        X_a_cv,
        a_cal_val[
            "target_presence"
        ],
    ),
    early_stopping_rounds=(
        A_PATIENCE
    ),
    use_best_model=True,
    verbose=False,
)

A_ITERATIONS = max(
    1,
    int(
        a_selector.get_best_iteration()
    )
    + 1,
)

a_cal_prob = (
    a_selector.predict_proba(
        X_a_cv
    )[:, 1]
)

(
    FINAL_THRESHOLD,
    final_threshold_search,
) = choose_threshold(
    a_cal_val[
        "target_presence"
    ],
    a_cal_prob,
)


(
    X_a_train,
    a_cat_idx,
) = prepare_X(
    train_dev,
    MODEL_A_NUMERIC,
)

(
    X_a_test,
    _,
) = prepare_X(
    final_test_features,
    MODEL_A_NUMERIC,
)


final_model_a = (
    CatBoostClassifier(
        iterations=A_ITERATIONS,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **A_PARAMS,
    )
)

final_model_a.fit(
    X_a_train,
    train_dev[
        "target_presence"
    ],
    cat_features=(
        a_cat_idx
    ),
    verbose=False,
)


final_presence_prob = (
    final_model_a.predict_proba(
        X_a_test
    )[:, 1]
)


print(
    "Model A iterations:",
    A_ITERATIONS,
)

print(
    "Frozen threshold:",
    FINAL_THRESHOLD,
)

print(
    "✅ Model A Test probabilities generated before label reveal."
)

Model A iterations: 324
Frozen threshold: 0.67
✅ Model A Test probabilities generated before label reveal.


# Part O. Frozen Conditional Model B 최종 학습

In [42]:
B_MAX_ITER = 2000
B_PATIENCE = 50

B_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}


positive_dev = (
    train_dev[
        train_dev[
            "target_presence"
        ].eq(1)
    ]
    .copy()
)

(
    b_cal_train,
    b_cal_val,
) = split_final_calibration(
    positive_dev
)

(
    X_b_ct,
    b_cat_idx,
) = prepare_X(
    b_cal_train,
    S3_NUMERIC,
)

(
    X_b_cv,
    _,
) = prepare_X(
    b_cal_val,
    S3_NUMERIC,
)


b_selector = (
    CatBoostRegressor(
        iterations=B_MAX_ITER,
        eval_metric="MAE",
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **B_PARAMS,
    )
)

b_selector.fit(
    X_b_ct,
    b_cal_train[
        "target_goals"
    ],
    cat_features=(
        b_cat_idx
    ),
    eval_set=(
        X_b_cv,
        b_cal_val[
            "target_goals"
        ],
    ),
    early_stopping_rounds=(
        B_PATIENCE
    ),
    use_best_model=True,
    verbose=False,
)

B_ITERATIONS = max(
    1,
    int(
        b_selector.get_best_iteration()
    )
    + 1,
)


(
    X_b_train,
    b_cat_idx,
) = prepare_X(
    positive_dev,
    S3_NUMERIC,
)

(
    X_b_test,
    _,
) = prepare_X(
    final_test_features,
    S3_NUMERIC,
)


final_model_b = (
    CatBoostRegressor(
        iterations=B_ITERATIONS,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **B_PARAMS,
    )
)

final_model_b.fit(
    X_b_train,
    positive_dev[
        "target_goals"
    ],
    cat_features=(
        b_cat_idx
    ),
    verbose=False,
)


final_conditional_pred = np.clip(
    final_model_b.predict(
        X_b_test
    ),
    0,
    None,
)


final_hard_pred = np.where(
    final_presence_prob
    >= FINAL_THRESHOLD,
    final_conditional_pred,
    0.0,
)


print(
    "Model B iterations:",
    B_ITERATIONS,
)

print(
    "✅ Hard Two-stage predictions frozen before Test label reveal."
)

Default metric period is 5 because MAE is/are not implemented for GPU


Model B iterations: 247
✅ Hard Two-stage predictions frozen before Test label reveal.


# Part P. Diagnostic comparator — Direct Regression

Direct Regression은 09-02에서 이미 Hard보다 열세였기 때문에
**Final model 후보가 아닙니다.**

Test에서 비교 수치만 기록합니다.
결과를 보고 최종 모델을 교체하지 않습니다.

In [43]:
D_MAX_ITER = 2000
D_PATIENCE = 50

D_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}


(
    d_cal_train,
    d_cal_val,
) = split_final_calibration(
    train_dev
)

(
    X_d_ct,
    d_cat_idx,
) = prepare_X(
    d_cal_train,
    DIRECT_NUMERIC,
)

(
    X_d_cv,
    _,
) = prepare_X(
    d_cal_val,
    DIRECT_NUMERIC,
)


d_selector = (
    CatBoostRegressor(
        iterations=D_MAX_ITER,
        eval_metric="MAE",
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **D_PARAMS,
    )
)

d_selector.fit(
    X_d_ct,
    d_cal_train[
        "target_goals"
    ],
    cat_features=(
        d_cat_idx
    ),
    eval_set=(
        X_d_cv,
        d_cal_val[
            "target_goals"
        ],
    ),
    early_stopping_rounds=(
        D_PATIENCE
    ),
    use_best_model=True,
    verbose=False,
)

D_ITERATIONS = max(
    1,
    int(
        d_selector.get_best_iteration()
    )
    + 1,
)


(
    X_d_train,
    d_cat_idx,
) = prepare_X(
    train_dev,
    DIRECT_NUMERIC,
)

(
    X_d_test,
    _,
) = prepare_X(
    final_test_features,
    DIRECT_NUMERIC,
)


direct_model = (
    CatBoostRegressor(
        iterations=D_ITERATIONS,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **D_PARAMS,
    )
)

direct_model.fit(
    X_d_train,
    train_dev[
        "target_goals"
    ],
    cat_features=(
        d_cat_idx
    ),
    verbose=False,
)

final_direct_pred = np.clip(
    direct_model.predict(
        X_d_test
    ),
    0,
    None,
)


print(
    "Direct comparator iterations:",
    D_ITERATIONS,
)

print(
    "✅ All predictions are now frozen."
)

Default metric period is 5 because MAE is/are not implemented for GPU


Direct comparator iterations: 320
✅ All predictions are now frozen.


# Part Q. FINAL TEST LABEL REVEAL

**여기서 처음으로 Test 정답을 다시 붙입니다.**

이 셀 아래의 결과가 프로젝트 Final Holdout 결과입니다.

In [44]:
final_predictions = (
    final_test_features[
        [
            "row_id",
            "player",
            "team",
            "league",
            "season",
            "target_season",
            "position_group",
            "goals",
            "changed_team_preseason",
            "transfer_event_preseason",
            "destination_team",
            "destination_in_big5",
            "market_value_known",
        ]
    ]
    .copy()
)


final_predictions[
    "presence_probability"
] = (
    final_presence_prob
)

final_predictions[
    "presence_threshold"
] = (
    FINAL_THRESHOLD
)

final_predictions[
    "conditional_goal_pred"
] = (
    final_conditional_pred
)

final_predictions[
    "hard_two_stage_pred"
] = (
    final_hard_pred
)

final_predictions[
    "direct_regression_pred"
] = (
    final_direct_pred
)


final_predictions = (
    final_predictions.merge(
        final_test_labels[
            [
                "row_id",
                "final_matched_next",
                "final_next_goals",
                "final_next_10plus",
            ]
        ],
        on="row_id",
        how="left",
        validate="one_to_one",
    )
)


assert final_predictions[
    "final_next_goals"
].notna().all()

assert final_predictions[
    "final_matched_next"
].notna().all()


final_predictions[
    "final_matched_next"
] = (
    final_predictions[
        "final_matched_next"
    ]
    .astype(bool)
)

final_predictions[
    "final_next_goals"
] = (
    final_predictions[
        "final_next_goals"
    ]
    .astype(float)
)


print(
    "🚨 FINAL TEST OPENED"
)

print(
    "Rows:",
    len(
        final_predictions
    ),
)

print(
    "Presence rate:",
    f"{final_predictions['final_matched_next'].mean():.2%}"
)

🚨 FINAL TEST OPENED
Rows: 926
Presence rate: 81.10%


# Part R. Final Overall Metrics

In [45]:
def regression_metrics(
    y,
    pred,
):
    y = np.asarray(
        y,
        dtype=float,
    )

    p = np.asarray(
        pred,
        dtype=float,
    )

    return {
        "mae": (
            mean_absolute_error(
                y,
                p,
            )
        ),
        "rmse": (
            mean_squared_error(
                y,
                p,
            )
            ** 0.5
        ),
        "r2": (
            r2_score(
                y,
                p,
            )
        ),
        "bias": float(
            np.mean(
                p - y
            )
        ),
    }


y_test = (
    final_predictions[
        "final_next_goals"
    ]
    .to_numpy(
        dtype=float
    )
)


METHODS = {
    "FINAL_HARD_TWO_STAGE": (
        final_predictions[
            "hard_two_stage_pred"
        ].to_numpy()
    ),

    "Conditional_No_Gate": (
        final_predictions[
            "conditional_goal_pred"
        ].to_numpy()
    ),

    "Direct_Regression_Diagnostic": (
        final_predictions[
            "direct_regression_pred"
        ].to_numpy()
    ),

    "Current_Goals_Naive": (
        final_predictions[
            "goals"
        ].to_numpy(
            dtype=float
        )
    ),
}


overall_rows = []

for method, pred in (
    METHODS.items()
):
    overall_rows.append({
        "method": method,
        **regression_metrics(
            y_test,
            pred,
        ),
    })


final_summary = (
    pd.DataFrame(
        overall_rows
    )
)


display(
    final_summary
    .sort_values(
        "mae"
    )
)

,method,mae,rmse,r2,bias
0,FINAL_HARD_TWO_STAGE,2.002956,3.098749,0.405859,0.326885
2,Direct_Regression_Diagnostic,2.066688,3.083756,0.411595,0.312186
1,Conditional_No_Gate,2.258825,3.213315,0.361114,0.683314
3,Current_Goals_Naive,2.755940,4.120224,-0.050408,1.298056


## Model A Final Test

In [46]:
presence_test = (
    final_predictions[
        "final_matched_next"
    ]
    .astype(int)
    .to_numpy()
)

prob_test = (
    final_predictions[
        "presence_probability"
    ]
    .to_numpy()
)

presence_pred = (
    prob_test
    >= FINAL_THRESHOLD
).astype(int)


(
    precision,
    recall,
    f1,
    support,
) = precision_recall_fscore_support(
    presence_test,
    presence_pred,
    labels=[
        0,
        1,
    ],
    zero_division=0,
)


model_a_final_metrics = pd.DataFrame([
    {
        "threshold": (
            FINAL_THRESHOLD
        ),
        "log_loss": (
            log_loss(
                presence_test,
                np.clip(
                    prob_test,
                    1e-7,
                    1 - 1e-7,
                ),
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "brier": (
            brier_score_loss(
                presence_test,
                prob_test,
            )
        ),
        "roc_auc": (
            roc_auc_score(
                presence_test,
                prob_test,
            )
        ),
        "pr_auc_exit": (
            average_precision_score(
                1 - presence_test,
                1 - prob_test,
            )
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                presence_test,
                presence_pred,
            )
        ),
        "macro_f1": (
            f1_score(
                presence_test,
                presence_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "exit_precision": (
            precision[0]
        ),
        "exit_recall": (
            recall[0]
        ),
        "exit_f1": (
            f1[0]
        ),
        "presence_precision": (
            precision[1]
        ),
        "presence_recall": (
            recall[1]
        ),
        "presence_f1": (
            f1[1]
        ),
    }
])


model_a_final_metrics

,threshold,log_loss,brier,roc_auc,pr_auc_exit,balanced_accuracy,macro_f1,exit_precision,exit_recall,exit_f1,presence_precision,presence_recall,presence_f1
0,0.67,0.278613,0.082627,0.906882,0.757476,0.808043,0.813516,0.712575,0.68,0.695906,0.926219,0.936085,0.931126


# Part S. Slice Analysis

In [47]:
def slice_row(
    name,
    frame,
    pred_col=(
        "hard_two_stage_pred"
    ),
):
    if frame.empty:
        return {
            "slice": name,
            "n": 0,
            "mae": np.nan,
            "bias": np.nan,
            "actual_mean": np.nan,
            "pred_mean": np.nan,
        }

    actual = (
        frame[
            "final_next_goals"
        ]
        .to_numpy(
            dtype=float
        )
    )

    pred = (
        frame[
            pred_col
        ]
        .to_numpy(
            dtype=float
        )
    )

    return {
        "slice": name,
        "n": len(
            frame
        ),
        "mae": (
            mean_absolute_error(
                actual,
                pred,
            )
        ),
        "bias": float(
            np.mean(
                pred - actual
            )
        ),
        "actual_mean": float(
            actual.mean()
        ),
        "pred_mean": float(
            pred.mean()
        ),
    }


slice_rows = []


slice_rows.append(
    slice_row(
        "ALL",
        final_predictions,
    )
)

slice_rows.append(
    slice_row(
        "ACTUAL_EXIT",
        final_predictions[
            final_predictions[
                "final_matched_next"
            ].eq(False)
        ],
    )
)

slice_rows.append(
    slice_row(
        "ACTUAL_PRESENCE",
        final_predictions[
            final_predictions[
                "final_matched_next"
            ].eq(True)
        ],
    )
)

slice_rows.append(
    slice_row(
        "CHANGED_TEAM_PRESEASON",
        final_predictions[
            final_predictions[
                "changed_team_preseason"
            ].eq(1)
        ],
    )
)

for position in [
    "FW",
    "MF",
]:
    slice_rows.append(
        slice_row(
            f"POSITION_{position}",
            final_predictions[
                final_predictions[
                    "position_group"
                ].eq(
                    position
                )
            ],
        )
    )

for league in sorted(
    final_predictions[
        "league"
    ]
    .dropna()
    .unique()
):
    slice_rows.append(
        slice_row(
            f"LEAGUE_{league}",
            final_predictions[
                final_predictions[
                    "league"
                ].eq(
                    league
                )
            ],
        )
    )


final_slice_summary = (
    pd.DataFrame(
        slice_rows
    )
)

display(
    final_slice_summary
)

,slice,n,mae,bias,actual_mean,pred_mean
0,ALL,926,2.002956,0.326885,2.844492,3.171378
1,ACTUAL_EXIT,175,0.904634,0.904634,0.000000,0.904634
2,ACTUAL_PRESENCE,751,2.258890,0.192257,3.507324,3.699580
3,CHANGED_TEAM_PRESEASON,150,1.740336,0.905556,1.866667,2.772223
4,POSITION_FW,491,2.653297,0.576250,3.975560,4.551810
5,POSITION_MF,435,1.268893,0.045418,1.567816,1.613234
6,LEAGUE_Bundesliga,167,1.834573,0.080421,3.023952,3.104373
7,LEAGUE_La Liga,211,1.909426,-0.011810,2.938389,2.926578
8,LEAGUE_Ligue 1,172,2.056066,0.488017,2.453488,2.941506
9,LEAGUE_Premier League,182,2.450435,0.632438,3.379121,4.011559


## High Scorer Slice

In [48]:
high_rows = []

for threshold in [
    10,
    15,
    20,
]:
    subset = (
        final_predictions[
            final_predictions[
                "final_next_goals"
            ].ge(
                threshold
            )
        ]
    )

    for method, col in {
        "FINAL_HARD_TWO_STAGE": (
            "hard_two_stage_pred"
        ),
        "Conditional_No_Gate": (
            "conditional_goal_pred"
        ),
        "Direct_Regression_Diagnostic": (
            "direct_regression_pred"
        ),
        "Current_Goals_Naive": (
            "goals"
        ),
    }.items():
        row = slice_row(
            f"{threshold}PLUS",
            subset,
            pred_col=(
                col
            ),
        )

        row[
            "threshold"
        ] = threshold

        row[
            "method"
        ] = method

        high_rows.append(
            row
        )


final_high_scorer_summary = (
    pd.DataFrame(
        high_rows
    )
)


display(
    final_high_scorer_summary[
        [
            "threshold",
            "method",
            "n",
            "mae",
            "bias",
            "actual_mean",
            "pred_mean",
        ]
    ]
)

,threshold,method,n,mae,bias,actual_mean,pred_mean
0,10,FINAL_HARD_TWO_STAGE,67,5.660453,-5.294845,13.805970,8.511125
1,10,Conditional_No_Gate,67,5.613360,-5.247752,13.805970,8.558218
2,10,Direct_Regression_Diagnostic,67,5.915322,-5.576253,13.805970,8.229717
3,10,Current_Goals_Naive,67,5.582090,-2.358209,13.805970,11.447761
4,15,FINAL_HARD_TWO_STAGE,22,7.171976,-6.994514,18.545455,11.550940
5,15,Conditional_No_Gate,22,7.171976,-6.994514,18.545455,11.550940
6,15,Direct_Regression_Diagnostic,22,7.316197,-7.313492,18.545455,11.231963
7,15,Current_Goals_Naive,22,6.454545,-4.272727,18.545455,14.272727
8,20,FINAL_HARD_TWO_STAGE,5,10.426166,-9.954014,26.400000,16.445986
9,20,Conditional_No_Gate,5,10.426166,-9.954014,26.400000,16.445986


# Part T. Error Analysis

In [49]:
final_predictions[
    "hard_error"
] = (
    final_predictions[
        "hard_two_stage_pred"
    ]
    - final_predictions[
        "final_next_goals"
    ]
)

final_predictions[
    "hard_abs_error"
] = (
    final_predictions[
        "hard_error"
    ].abs()
)


top_under = (
    final_predictions
    .sort_values(
        "hard_error"
    )
    .head(25)
    .copy()
)

top_over = (
    final_predictions
    .sort_values(
        "hard_error",
        ascending=False,
    )
    .head(25)
    .copy()
)


print(
    "Top Underpredictions"
)

display(
    top_under[
        [
            "player",
            "team",
            "final_next_goals",
            "hard_two_stage_pred",
            "presence_probability",
            "conditional_goal_pred",
            "hard_error",
        ]
    ]
)


print(
    "Top Overpredictions"
)

display(
    top_over[
        [
            "player",
            "team",
            "final_next_goals",
            "hard_two_stage_pred",
            "presence_probability",
            "conditional_goal_pred",
            "hard_error",
        ]
    ]
)

Top Underpredictions


,player,team,final_next_goals,hard_two_stage_pred,presence_probability,conditional_goal_pred,hard_error
869,Vedat Muriqi,Mallorca,23.0,5.247175,0.925656,5.247175,-17.752825
269,Esteban Lepaul,Angers,21.0,5.480939,0.989443,5.480939,-15.519061
235,Donyell Malen,Dortmund,18.0,5.726864,0.871576,5.726864,-12.273136
336,Harry Kane,Bayern Munich,36.0,23.891879,0.986226,23.891879,-12.108121
216,Deniz Undav,Stuttgart,19.0,7.463717,0.968390,7.463717,-11.536283
854,Toni Martínez,Alavés,14.0,3.001388,0.785556,3.001388,-10.998612
174,Christoph Baumgartner,RB Leipzig,13.0,2.328314,0.965950,2.328314,-10.671686
230,Dominic Calvert-Lewin,Everton,14.0,3.465722,0.930294,3.465722,-10.534278
631,Morgan Gibbs-White,Nott'ham Forest,15.0,4.880528,0.993889,4.880528,-10.119472
22,Akor Adams,Montpellier,10.0,0.000000,0.417095,3.155227,-10.000000


Top Overpredictions


,player,team,final_next_goals,hard_two_stage_pred,presence_probability,conditional_goal_pred,hard_error
627,Mohamed Salah,Liverpool,7.0,22.335115,0.986853,22.335115,15.335115
42,Alexander Isak,Newcastle Utd,3.0,17.307340,0.994777,17.307340,14.307340
681,Omar Marmoush,Eint Frankfurt,3.0,13.827964,0.994474,13.827964,10.827964
44,Alexandre Lacazette,Lyon,0.0,10.785541,0.932269,10.785541,10.785541
904,Yoane Wissa,Brentford,1.0,11.064048,0.981637,11.064048,10.064048
769,Romelu Lukaku,Napoli,1.0,10.726543,0.975281,10.726543,9.726543
525,Loïs Openda,RB Leipzig,1.0,10.423452,0.988440,10.423452,9.423452
845,Tim Kleindienst,Gladbach,0.0,9.237049,0.988762,9.237049,9.237049
168,Chris Wood,Nott'ham Forest,3.0,11.757285,0.985641,11.757285,8.757285
736,Randal Kolo Muani,Juventus,1.0,9.259015,0.804341,9.259015,8.259015


## Hard Gate False Exit

실제 다음 시즌 record가 존재하는데
Model A가 threshold 아래로 내려 0골 처리한 사례입니다.

In [50]:
false_exit = (
    final_predictions[
        final_predictions[
            "final_matched_next"
        ].eq(True)
        & (
            final_predictions[
                "presence_probability"
            ]
            < FINAL_THRESHOLD
        )
    ]
    .copy()
    .sort_values(
        "final_next_goals",
        ascending=False,
    )
)


display(
    false_exit[
        [
            "player",
            "team",
            "final_next_goals",
            "presence_probability",
            "presence_threshold",
            "conditional_goal_pred",
            "hard_two_stage_pred",
        ]
    ]
    .head(50)
)

,player,team,final_next_goals,presence_probability,presence_threshold,conditional_goal_pred,hard_two_stage_pred
22,Akor Adams,Montpellier,10.0,0.417095,0.67,3.155227,0.0
387,Jamie Vardy,Leicester City,7.0,0.090305,0.67,5.629770,0.0
520,Lorenzo Colombo,Empoli,7.0,0.572640,0.67,4.519703,0.0
150,Budu Zivzivadze,Heidenheim,6.0,0.317472,0.67,1.698182,0.0
640,Musa Al-Taamari,Montpellier,6.0,0.224620,0.67,2.601216,0.0
132,Bilal El Khannouss,Leicester City,4.0,0.423251,0.67,2.183289,0.0
457,Julio Enciso,Ipswich Town,3.0,0.394538,0.67,2.373568,0.0
170,Christian Eriksen,Manchester Utd,3.0,0.190321,0.67,1.152373,0.0
511,Lesley Ugochukwu,Southampton,3.0,0.465369,0.67,1.101686,0.0
585,Mateus Fernandes,Southampton,3.0,0.625036,0.67,1.410915,0.0


# Part U. Feature Coverage on Final Test

Final Test 성능이 낮을 경우
2025 preseason external-data coverage가 개발 시기와 달라졌는지 확인하기 위한 표입니다.

In [51]:
coverage_rows = [
    {
        "metric": (
            "final_test_rows"
        ),
        "value": len(
            final_predictions
        ),
    },
    {
        "metric": (
            "tm_player_id_coverage"
        ),
        "value": (
            final_test_features[
                "tm_player_id"
            ]
            .notna()
            .mean()
        ),
    },
    {
        "metric": (
            "market_value_coverage"
        ),
        "value": (
            final_test_features[
                "market_value_known"
            ]
            .mean()
        ),
    },
    {
        "metric": (
            "preseason_transfer_rate"
        ),
        "value": (
            final_test_features[
                "transfer_event_preseason"
            ]
            .mean()
        ),
    },
    {
        "metric": (
            "changed_team_rate"
        ),
        "value": (
            final_test_features[
                "changed_team_preseason"
            ]
            .mean()
        ),
    },
    {
        "metric": (
            "new_team_strength_missing_rate"
        ),
        "value": (
            final_test_features[
                "new_team_strength_missing"
            ]
            .mean()
        ),
    },
]


final_feature_coverage = (
    pd.DataFrame(
        coverage_rows
    )
)

final_feature_coverage

,metric,value
0,final_test_rows,926.000000
1,tm_player_id_coverage,0.939525
2,market_value_coverage,0.934125
3,preseason_transfer_rate,0.192225
4,changed_team_rate,0.161987
5,new_team_strength_missing_rate,0.096112


# Part V. 저장

In [54]:
import json

OUTPUTS = {
    "summary": (
        ARTIFACT_DIR
        / "11_final_test_summary.csv"
    ),
    "model_a": (
        ARTIFACT_DIR
        / "11_final_test_model_a_metrics.csv"
    ),
    "slice": (
        ARTIFACT_DIR
        / "11_final_test_slice_summary.csv"
    ),
    "high_scorer": (
        ARTIFACT_DIR
        / "11_final_test_high_scorer_summary.csv"
    ),
    "predictions": (
        ARTIFACT_DIR
        / "11_final_test_predictions.csv"
    ),
    "top_under": (
        ARTIFACT_DIR
        / "11_final_test_top_underpredictions.csv"
    ),
    "top_over": (
        ARTIFACT_DIR
        / "11_final_test_top_overpredictions.csv"
    ),
    "false_exit": (
        ARTIFACT_DIR
        / "11_final_test_false_exit_cases.csv"
    ),
    "feature_coverage": (
        ARTIFACT_DIR
        / "11_final_test_feature_coverage.csv"
    ),
    "threshold_search": (
        ARTIFACT_DIR
        / "11_final_threshold_search.csv"
    ),
    "test_snapshot": (
        ARTIFACT_DIR
        / "11_final_test_preseason_snapshot.csv"
    ),
}


final_summary.to_csv(
    OUTPUTS[
        "summary"
    ],
    index=False,
)

model_a_final_metrics.to_csv(
    OUTPUTS[
        "model_a"
    ],
    index=False,
)

final_slice_summary.to_csv(
    OUTPUTS[
        "slice"
    ],
    index=False,
)

final_high_scorer_summary.to_csv(
    OUTPUTS[
        "high_scorer"
    ],
    index=False,
)

final_predictions.to_csv(
    OUTPUTS[
        "predictions"
    ],
    index=False,
)

top_under.to_csv(
    OUTPUTS[
        "top_under"
    ],
    index=False,
)

top_over.to_csv(
    OUTPUTS[
        "top_over"
    ],
    index=False,
)

false_exit.to_csv(
    OUTPUTS[
        "false_exit"
    ],
    index=False,
)

final_feature_coverage.to_csv(
    OUTPUTS[
        "feature_coverage"
    ],
    index=False,
)

final_threshold_search.to_csv(
    OUTPUTS[
        "threshold_search"
    ],
    index=False,
)

final_test_features.to_csv(
    OUTPUTS[
        "test_snapshot"
    ],
    index=False,
)


PROTOCOL = {
    "stage": (
        "11 Final Test + Slice/Error Analysis"
    ),

    "selection_status": (
        "FROZEN BEFORE TEST"
    ),

    "feature_set": (
        "FULL"
    ),

    "feature_stability_result": (
        "KEEP_FULL"
    ),

    "development_label": (
        "09-01A conservative audited labels"
    ),

    "final_test_label": (
        "original locked test.csv labels; "
        "no post-hoc correction"
    ),

    "model_a": {
        "type": (
            "Fixed CatBoost Classifier"
        ),
        "iterations": (
            A_ITERATIONS
        ),
        "threshold": (
            FINAL_THRESHOLD
        ),
        "threshold_source": (
            "last development input season "
            "before final test"
        ),
    },

    "model_b": {
        "type": (
            "Fixed CatBoost Regressor S3"
        ),
        "iterations": (
            B_ITERATIONS
        ),
        "training_population": (
            "matched_next_audited == True"
        ),
    },

    "integration": (
        "Hard Gate"
    ),

    "test_input_season": (
        "2024-2025"
    ),

    "test_target_season": (
        "2025-2026"
    ),

    "test_opened": (
        True
    ),

    "test_used_for_selection": (
        False
    ),

    "post_test_model_change_allowed": (
        False
    ),
}


PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "11_final_test_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:"
)

for name, path in (
    OUTPUTS.items()
):
    print(
        f"- {name:<18}",
        path,
    )

print(
    "- protocol          ",
    PROTOCOL_PATH,
)

Saved:
- summary            /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_summary.csv
- model_a            /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_model_a_metrics.csv
- slice              /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_slice_summary.csv
- high_scorer        /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_high_scorer_summary.csv
- predictions        /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_predictions.csv
- top_under          /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_top_underpredictions.csv
- top_over           /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_top_overpredictions.csv
- false_exit         /content/drive/MyDrive/next_season_goal_prediction/artifacts/11_final_test_false_exit_cases.csv
- feature_coverage   /content/drive/MyDrive/next_season_goal_pred

# 실행 후 보내줄 파일

우선 아래 7개면 최종 결론을 낼 수 있습니다.

```text
11_final_test_summary.csv
11_final_test_model_a_metrics.csv
11_final_test_slice_summary.csv
11_final_test_high_scorer_summary.csv
11_final_test_predictions.csv
11_final_test_feature_coverage.csv
11_final_test_protocol.json
```

이 결과는 더 이상 모델 선택에 쓰지 않고
**프로젝트 최종 일반화 성능**으로 해석합니다.